## 1. Slowly Changing Dimensions (SCD) - comparison of announced Types

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice. 

Each type answers the question: 
- when an attribute value changes?
- what do we do with the old value?

---

| Type | Name | Strategy | Year announced | Old value preserved? | Full history? | New objects created |
|------|------|---------|---------------|---------------------|---------------|--------------------|
| 0 | Retain / Original | Never change the value | 1996 (2008*) | ✅ (never overwritten) | ❌ | None — no schema or table changes |
| 1 | Overwrite | Replace old with new | 1996 | ❌ | ❌ | None — same table, same columns, in-place updates |
| 2 | Add New Row | Insert a new row, close out the old | 1996 | ✅ | ✅ | **New columns** added to the existing dimension: `valid_from`, `valid_to`, `is_current`, plus a **surrogate key** column (e.g. `product_sk`) |
| 3 | Add New Column | Keep old + new in separate columns | 1996 | ✅ (one prior version) | Partial (current + previous) | **New columns** added to the existing dimension (e.g. `previous_category`) — one per tracked attribute |
| 4 | Mini-Dimension | Rapidly changing attrs split into a separate dimension | 2002 | N/A (separate table) | ✅ (via mini-dim grouping) | **New table** (mini-dimension) for the rapidly-changing attributes, keyed by a surrogate key; base dimension gains a **new FK column** (e.g. `mini_sk`) pointing to it |
| 5 | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | 2002 | Partial | ✅ (via mini-dim) | Everything from Type 4 **plus new outrigger columns** copied into the base dimension (e.g. `current_category`, `current_brand`) carrying the latest mini-dim values |
| 6 | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | 2002 | Partial | ✅ | All Type 2 columns **plus a new current-value column** (e.g. `current_category`) stamped across every history row |
| 7 | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | 2013 | Partial | ✅ | **Second physical table** — a Type 1 snapshot table alongside the Type 2 history table (two tables maintained in parallel) |

---

*announced in 1996 with other concepts, but described in details in 2008.

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

> **New objects:** None. The dimension table and its schema stay exactly as-is; no columns, tables, or lookup structures are added.
>
> **Announced:** Formally described by the Kimball Group in the third edition of *The Data Warehouse Toolkit* (2008), complementing the original three types.
> **Data Warehouse use case:** Preserving immutable attributes such as a customer's original registration date or date of birth that must never be modified.
> **Data Lakehouse use case:** Retaining original source-system landing timestamps in the Bronze layer for full audit traceability across the medallion architecture.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

> **New objects:** None. Same table, same columns — rows are simply overwritten in place.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Correcting a misspelled customer name or keeping the latest shipping address when only the current value matters.
> **Data Lakehouse use case:** Overwriting the latest record state in the Silver/Gold Delta tables via `MERGE` for downstream consumers that need only current values.

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

> **New objects:** The **existing dimension table gains new columns** — at minimum a surrogate key (e.g. `product_sk`), `valid_from`, `valid_to`, and `is_current`. No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Tracking full customer address history so that fact rows join to the dimension version that was active at the time of each transaction.
> **Data Lakehouse use case:** Maintaining full version history in Delta tables using Change Data Feed (CDF) with `MERGE` SCD Type 2 logic for point-in-time analytics and incremental CDC pipelines.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

> **New objects:** **New columns** added to the existing dimension table — one `previous_*` column per tracked attribute (e.g. `previous_category`). No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Comparing current vs. previous sales-territory assignment for an employee without the overhead of full history rows.
> **Data Lakehouse use case:** Lightweight one-level history tracking in a Silver Delta table when full SCD Type 2 is unnecessary but a before/after comparison is still required.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

> **New objects:** A **new lookup table** (the mini-dimension) holding one row per unique attribute combination, keyed by its own surrogate key. The base dimension gains a **new foreign-key column** (e.g. `mini_sk`) referencing the mini-dimension.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) to handle rapidly changing attributes.
> **Data Warehouse use case:** Managing fast-changing customer demographics (income band, credit score, dependents) without exploding the base customer dimension.
> **Data Lakehouse use case:** Splitting volatile attributes into a separate Delta table keyed by surrogate key, referenced from the main dimension to keep the base table compact and queryable.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

> **New objects:** Everything from Type 4 (new mini-dimension table + `mini_sk` FK on the base) **plus new outrigger columns** on the base dimension (e.g. `current_category`, `current_brand`) that carry a Type 1 copy of the latest mini-dimension attribute values.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as an extension of the Type 4 mini-dimension pattern.
> **Data Warehouse use case:** Allowing BI dashboards to filter by the current income band directly from the base dimension while the mini-dimension preserves full historical accuracy.
> **Data Lakehouse use case:** Adding current-value outrigger columns in the Gold layer so BI tools can query volatile attributes without joining to the mini-dimension Delta table.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

> **New objects:** All Type 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`) **plus a new current-value column** (e.g. `current_category`) added to the existing dimension table. No additional tables are created.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as a hybrid combining Types 1 and 2.
> **Data Warehouse use case:** Grouping all historical sales by the *current* product category while still preserving the category value as it was at each point in time.
> **Data Lakehouse use case:** Stamping a `current_*` column across every history row in a Delta table so a single Gold-layer query supports both current and point-in-time analytics.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

> **New objects:** A **second physical table** — a Type 1 snapshot table alongside the Type 2 history table. The Type 2 table carries the SCD 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`); the Type 1 table mirrors the original dimension schema. Two tables must be maintained and kept in sync.
>
> **Announced:** Type 7 was formally described in 2013 by the Kimball Group. It was included in the 3rd edition of The Data Warehouse Toolkit the same year.
> **Data Warehouse use case:** Serving a Type 1 current-state table for fast dashboards and a Type 2 history table for accurate point-in-time fact joins from the same source feed.
> **Data Lakehouse use case:** Maintaining a current-state Delta table (feeding BI/SQL warehouses) alongside a full-history Delta table (feeding incremental CDC and audit pipelines), both refreshed from the same batch or streaming source.




### 0. Parameters — catalog, schema and table names

Everything this notebook creates lives in one schema, and its name is set here. Change the two widget values to redirect every table the notebook creates (MERGE examples, schema-evolution MERGE, stress test, clean-up) to a different catalog and schema.

This cell registers the two widgets, creates the schema and sets it as the session default (`USE CATALOG` / `USE SCHEMA`). The `%sql` cells below therefore address their tables unqualified (for example `scd_t1_dim_product`), while the Python cells use `catalog_name`, `schema_name` and the `TBL_*` variables derived from the same widgets.

Run this cell first, before any cell that writes a table.


In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "scd_concept")

catalog_name = dbutils.widgets.get("catalog")
schema_name  = dbutils.widgets.get("schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# Make every UNQUALIFIED table name in the %sql cells below resolve here:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# Full table names used by the Python cells and by the clean-up cell at the end
TBL_T0            = f"{catalog_name}.{schema_name}.scd_t0_dim_product"
TBL_T1            = f"{catalog_name}.{schema_name}.scd_t1_dim_product"
TBL_T2            = f"{catalog_name}.{schema_name}.scd_t2_dim_product"
TBL_T3            = f"{catalog_name}.{schema_name}.scd_t3_dim_product"
TBL_T4_MINI       = f"{catalog_name}.{schema_name}.scd_t4_mini_dim"
TBL_T4_BASE       = f"{catalog_name}.{schema_name}.scd_t4_base_dim"
TBL_T5_MINI       = f"{catalog_name}.{schema_name}.scd_t5_mini_dim"
TBL_T5_BASE       = f"{catalog_name}.{schema_name}.scd_t5_base_dim"
TBL_T6            = f"{catalog_name}.{schema_name}.scd_t6_dim_product"
TBL_T6_CURRENT    = f"{catalog_name}.{schema_name}.scd_t6_current_values"
TBL_T7_CURRENT    = f"{catalog_name}.{schema_name}.scd_t7_dim_current"
TBL_T7_HISTORY    = f"{catalog_name}.{schema_name}.scd_t7_dim_history"
TBL_T8            = f"{catalog_name}.{schema_name}.scd_t8_dim_product"
TBL_T8_EVOLUTION  = f"{catalog_name}.{schema_name}.scd_t8_evolution"
TBL_T8_STRESS     = f"{catalog_name}.{schema_name}.scd_t8_stress"

ALL_TABLES = [TBL_T0, TBL_T1, TBL_T2, TBL_T3, TBL_T4_MINI, TBL_T4_BASE,
              TBL_T5_MINI, TBL_T5_BASE, TBL_T6, TBL_T6_CURRENT,
              TBL_T7_CURRENT, TBL_T7_HISTORY, TBL_T8, TBL_T8_EVOLUTION,
              TBL_T8_STRESS]

print(f"All tables will be created in: {catalog_name}.{schema_name}")
print(f"Registered table names ({len(ALL_TABLES)}):")
for t in ALL_TABLES:
    print(f"  - {t}")

In [0]:
# =====================================================================
# 0. IMPORTS — every install / import statement of the notebook, in one place
# =====================================================================
# All libraries used anywhere below are imported here, right after the
# parameters cell, so no later cell carries its own import block.
# Nothing needs to be installed: every module below ships with the
# Databricks runtime (no %pip step is required).

import random
import time
from functools import reduce

from pyspark.sql import Row
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Imports ready: random, time, functools.reduce | pyspark.sql: Row, DataFrame, F (functions), Window")


### Type 2 example (most common in practice)

Using the `dim_product` table above, if product 3 changes from category *Kitchen* to *Home*, a Type 2 table would look like:

| product_sk (surrogate) | product_id | product_name | category | valid_from | valid_to | is_current |
|------------------------|------------|--------------|----------|------------|----------|------------|
| 301 | 3 | Coffee Mug | Kitchen | 2024-01-01 | 2024-06-30 | false |
| 302 | 3 | Coffee Mug | Home | 2024-07-01 | 9999-12-31 | true |

The surrogate key `product_sk` is unique per version; the natural key `product_id` repeats. Fact rows reference `product_sk` so each fact is associated with the correct version of the product.

> **Reference:** [Slowly changing dimension — Wikipedia](https://en.wikipedia.org/wiki/Slowly_changing_dimension)

## 2. Sample Dimension table

In [0]:
dim_product = spark.createDataFrame([
    Row(product_id=1, product_name="Laptop",    category="Electronics", brand="Acme",     color="Silver"),
    Row(product_id=2, product_name="Headphones", category="Electronics", brand="Bose",     color="Black"),
    Row(product_id=3, product_name="Coffee Mug", category="Kitchen",     brand="Ikea",     color="White"),
    Row(product_id=4, product_name="Notebook",    category="Office",      brand="Moleskine", color="Red"),
    Row(product_id=5, product_name="Desk Lamp",   category="Office",      brand="Ikea",     color="Brass"),
])

dim_product.display()

## 3. Simulate incoming data batch

### Simulation of incoming data (change feed from the source system)

60 change events (10x the original batch): one feed, but carrying multiple change points spread over several months.

* insert → brand-new products (ids 6..22)
* update → full new snapshot of the record; the same `product_id` can appear many times with different `change_dates` (e.g. 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 20)
* delete → product removed from the source (only the key matters)

Every SCD type below consumes this same feed, each in its own way. **Note:** with multiple events per product, downstream logic typically has to pick the latest event per product (or replay events in date order) rather than blindly joining on `product_id`.


In [0]:
change_events = [
    # --- 2024-07-01: initial batch -----------------------------------
    # inserts: brand-new products
    (6,  "Mechanical Keyboard", "Electronics", "Logitech",       "Black",       "insert", "2024-07-01"),
    (7,  "Wall Clock",          "Home",        "Ikea",            "White",       "insert", "2024-07-01"),
    (8,  "Smart Watch",         "Electronics", "Garmin",          "Slate",       "insert", "2024-07-01"),
    (9,  "Yoga Mat",            "Sports",      "Decathlon",       "Blue",        "insert", "2024-07-01"),
    (10, "Blender",             "Kitchen",     "Philips",         "White",       "insert", "2024-07-01"),
    (11, "Office Chair",        "Office",      "Herman Miller",   "Black",       "insert", "2024-07-01"),
    (12, "Desk Organizer",      "Office",      "Knoll",            "Grey",        "insert", "2024-07-01"),
    (13, "Water Bottle",        "Sports",      "CamelBak",         "Cyan",        "insert", "2024-07-01"),
    (14, "Air Purifier",        "Home",        "Dyson",            "Silver",      "insert", "2024-07-01"),
    (15, "Gaming Mouse",        "Electronics", "Razer",            "Green",       "insert", "2024-07-01"),
    (16, "Standing Desk",       "Office",      "Varidesk",         "Oak",         "insert", "2024-07-01"),
    (17, "Curtains",            "Home",        "Zara Home",        "Beige",       "insert", "2024-07-01"),
    (18, "Coffee Grinder",      "Kitchen",     "Braun",            "Steel",       "insert", "2024-07-01"),
    (19, "Backpack",            "Travel",      "Samsonite",        "Red",         "insert", "2024-07-01"),
    (20, "Umbrella",            "Home",        "Totes",            "Navy",        "insert", "2024-07-01"),
    # updates: full new snapshot of existing products
    (1,  "Laptop",              "Computers",   "Acme",             "Silver",      "update", "2024-07-01"),
    (2,  "Headphones",          "Electronics", "Sony",             "Black",       "update", "2024-07-01"),
    (3,  "Coffee Mug",          "Home",        "Ikea",             "White",       "update", "2024-07-01"),

    # --- August: first wave of follow-up changes ----------------------
    (5,  "Desk Lamp",           "Lighting",    "Ikea",             "Brass",       "update", "2024-08-05"),
    (2,  "Headphones",          "Electronics", "Sony",             "Beige",       "update", "2024-08-12"),  # 2nd change for product 2
    (6,  "Mechanical Keyboard", "Electronics", "Razer",            "Black",       "update", "2024-08-20"),
    (1,  "Laptop",              "Computers",   "Acme",             "Graphite",    "update", "2024-08-25"),  # 2nd change for product 1
    (10, "Blender",             "Appliances",  "Philips",          "White",       "update", "2024-08-28"),
    (21, "Drone",               "Electronics", "DJI",              "Grey",        "insert", "2024-08-30"),  # late insert

    # --- September -----------------------------------------------------
    (12, None, None, None, None, "delete", "2024-09-10"),         # product 12 removed
    (2,  "Headphones",          "Audio",       "Sony",             "Beige",       "update", "2024-09-15"),  # 3rd change for product 2
    (7,  "Wall Clock",          "Decor",       "Ikea",             "White",       "update", "2024-09-18"),
    (3,  "Coffee Mug",          "Home",        "Ikea",             "Ivory",       "update", "2024-09-22"),  # 2nd change for product 3
    (22, "Bluetooth Speaker",   "Electronics", "Bose",             "Black",       "insert", "2024-09-25"),
    (5,  "Desk Lamp",           "Lighting",    "Philips",          "Brass",       "update", "2024-09-28"),  # 2nd change for product 5
    (1,  "Laptop Pro",          "Computers",   "Acme",             "Graphite",    "update", "2024-09-30"),  # renamed
    (8,  "Smart Watch",         "Electronics", "Garmin",           "Black",       "update", "2024-09-30"),

    # --- October -------------------------------------------------------
    (10, "Blender",             "Appliances",  "Philips",          "Cream",       "update", "2024-10-05"),
    (4,  "Notebook",            "Office",      "Moleskine",        "Black",       "update", "2024-10-08"),
    (20, "Umbrella",            "Home",        "Totes",            "Green",       "update", "2024-10-12"),
    (17, "Curtains",            "Home",        "Zara Home",        "White",       "update", "2024-10-15"),
    (9,  "Yoga Mat",            "Sports",      "Decathlon",        "Purple",      "update", "2024-10-18"),
    (2,  "Headphones",          "Audio",       "Sony",             "Navy",        "update", "2024-10-22"),  # 4th change for product 2
    (13, "Water Bottle",        "Sports",      "CamelBak",         "Charcoal",    "update", "2024-10-25"),
    (14, "Air Purifier",        "Appliances",  "Dyson",            "Silver",      "update", "2024-10-28"),
    (4,  None, None, None, None, "delete", "2024-10-31"),         # updated earlier in Oct, removed at month end

    # --- November ------------------------------------------------------
    (6,  "Mechanical Keyboard", "Electronics", "Razer",            "White",       "update", "2024-11-04"),  # 2nd change for product 6
    (11, "Office Chair",        "Office",      "Herman Miller",    "Grey",        "update", "2024-11-08"),
    (1,  "Laptop Pro",          "Computers",   "Acer",             "Graphite",    "update", "2024-11-12"),  # 4th change for product 1
    (16, "Standing Desk",       "Office",      "Varidesk",         "Walnut",      "update", "2024-11-15"),
    (10, "Blender",             "Appliances",  "Vitamix",          "Cream",       "update", "2024-11-18"),  # 3rd change for product 10
    (14, None, None, None, None, "delete", "2024-11-20"),         # product 14 removed
    (19, "Backpack",            "Travel",      "Samsonite",        "Black",       "update", "2024-11-22"),
    (5,  "Desk Lamp",           "Lighting",    "Philips",          "Matte Black",  "update", "2024-11-25"),  # 3rd change for product 5
    (8,  "Smart Watch",         "Electronics", "Apple",            "Black",       "update", "2024-11-28"),  # 2nd change for product 8

    # --- December ------------------------------------------------------
    (2,  "Headphones",          "Audio",       "Sennheiser",       "Navy",        "update", "2024-12-02"),  # 5th change for product 2
    (7,  "Wall Clock",          "Decor",       "Ikea",             "Oak",         "update", "2024-12-05"),  # 2nd change for product 7
    (15, "Gaming Mouse",        "Electronics", "Razer",            "Pink",        "update", "2024-12-08"),
    (22, "Bluetooth Speaker",   "Electronics", "Sonos",            "Black",       "update", "2024-12-10"),
    (9,  "Yoga Mat",            "Sports",      "Lululemon",        "Purple",      "update", "2024-12-12"),  # 2nd change for product 9
    (3,  "Coffee Mug",          "Home",        "Bodum",            "Ivory",       "update", "2024-12-15"),  # 3rd change for product 3
    (18, "Coffee Grinder",      "Kitchen",     "Baratza",          "Steel",       "update", "2024-12-18"),
    (11, "Office Chair",        "Office",      "Steelcase",        "Grey",        "update", "2024-12-20"),  # 2nd change for product 11
    (17, None, None, None, None, "delete", "2024-12-22"),         # product 17 removed
    (20, "Umbrella",            "Home",        "Windbrella",       "Green",       "update", "2024-12-28"),  # 2nd change for product 20
]

changes = spark.createDataFrame(
    change_events,
    """product_id long, product_name string, category string,
       brand string, color string, change_type string, change_date string""",
).withColumn("change_date", F.to_date("change_date")).orderBy("change_date", "product_id")

display(changes)

events_with_initial = (
    dim_product
    .select("product_id", "product_name", "category", "brand", "color")
    .withColumn("change_type", F.lit("initial"))
    .withColumn("change_date", F.to_date(F.lit("2024-01-01")))
    .unionByName(
        changes.select("product_id", "product_name", "category", "brand", "color", "change_type", "change_date")
    )
)

latest_events = (
    events_with_initial
    .withColumn("rn", F.row_number().over(Window.partitionBy("product_id").orderBy(F.col("change_date").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

## 4. Behaviour of each SCD Type

### Type 0 — retain (original value)

* inserts → applied (new rows added once, then never touched again)
* updates → ignored (original value kept forever)
* deletes → ignored (row stays; Type 0 never changes anything)


In [0]:
inserts = changes.filter(F.col("change_type") == "insert")

dim_product_t0 = dim_product.unionByName(
    inserts.select("product_id", "product_name", "category", "brand", "color")
)

display(dim_product_t0.orderBy("product_id"))

### Type 1 — overwrite

* inserts → added as new rows
* updates → existing row overwritten in place (old value lost)
* deletes → row removed


In [0]:
dim_product_t1 = (
    latest_events
    .filter(F.col("change_type") != "delete")
    .select("product_id", "product_name", "category", "brand", "color")
)

display(dim_product_t1.orderBy("product_id"))

### Type 2 — add new row (full history)

* inserts → new current rows
* updates → old version closed (`valid_to` = `change_date` - 1, `is_current` = false) + new version inserted with a fresh surrogate key
* deletes → old version closed, no replacement row


In [0]:
w_t2 = Window.partitionBy("product_id").orderBy("change_date")

t2_versions = (
    events_with_initial
    .withColumn("next_date", F.lead("change_date").over(w_t2))
    .filter(F.col("change_type") != "delete")
    .withColumn("valid_from", F.col("change_date"))
    .withColumn("valid_to", F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))))
    .withColumn("is_current", F.col("next_date").isNull())
)

dim_product_t2_final = (
    t2_versions
    .withColumn("product_sk", (F.row_number().over(Window.orderBy("product_id", "valid_from")) + F.lit(100)).cast("long"))
    .select("product_id", "product_name", "category", "brand", "color", "product_sk", "valid_from", "valid_to", "is_current")
)

display(dim_product_t2_final.orderBy("product_id", "product_sk"))
display(dim_product_t2_final.filter(F.col("is_current") == True) \
    .orderBy("product_id", "product_sk") \
    .drop("is_current")
)

### Type 3 — add new column (current + previous)

* inserts → added as new rows (no previous value yet)
* updates → old value moved to `previous_category`, new value becomes current
* deletes → row removed


In [0]:
current_t3 = (
    latest_events
    .filter(F.col("change_type") != "delete")
    .select("product_id", "product_name", "category", "brand", "color")
)

previous_t3 = (
    events_with_initial
    .filter(F.col("change_type") != "delete")
    .join(current_t3.select("product_id", F.col("category").alias("current_category")), "product_id")
    .filter(F.col("category") != F.col("current_category"))
    .groupBy("product_id")
    .agg(F.max_by("category", "change_date").alias("previous_category"))
)

dim_product_t3 = current_t3.join(previous_t3, "product_id", "left")
display(dim_product_t3.orderBy("product_id"))

### Type 4 — mini-dimension

Rapidly changing attributes (category, brand, color) are split out of the base dimension into a mini-dimension: one row per unique attribute combination. The base dimension keeps only stable attributes plus a reference (`mini_sk`) to the current group — that reference is overwritten Type 1-style on change. History lives in the fact table's `mini_sk` references, not in the dimension itself.

* inserts → new group(s) in the mini-dim + new base row pointing at them
* updates → base row re-pointed to the (possibly new) matching group
* deletes → base row removed; groups stay (historical facts still use them)


In [0]:
all_combos = (
    events_with_initial
    .filter(F.col("change_type") != "delete")
    .select("category", "brand", "color")
    .distinct()
)

mini_dim_t4 = all_combos.withColumn(
    "mini_sk", (F.row_number().over(Window.orderBy("category", "brand", "color")) + F.lit(1000)).cast("long")
)

w_first = Window.partitionBy("product_id").orderBy("change_date")
stable_names = (
    events_with_initial
    .filter(F.col("change_type") != "delete")
    .withColumn("rn", F.row_number().over(w_first))
    .filter(F.col("rn") == 1)
    .select("product_id", "product_name")
)

base_dim_t4_final = (
    latest_events
    .filter(F.col("change_type") != "delete")
    .select("product_id", "category", "brand", "color")
    .join(mini_dim_t4, ["category", "brand", "color"])
    .join(stable_names, "product_id")
    .select("product_id", "product_name", "mini_sk")
)

print("Mini-dimension (one row per unique attribute combination):")
display(mini_dim_t4.orderBy("mini_sk"))
print("Base dimension (stable attributes + current group reference):")
display(base_dim_t4_final.orderBy("product_id"))

### Type 5 — mini-dimension + Type 1 outrigger

Type 4 (cell above) plus the base dimension carries a Type 1 copy of the key mini-dimension attributes as an "outrigger" — overwritten on change. This lets you group/filter by the current value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history via `mini_sk` references.

Requires the Type 4 cell above to have been executed (uses `base_dim_t4_final` and `mini_dim_t4`).


### Outrigger

The outrigger holds the current group attributes copied into the base dimension. In a real warehouse these columns are stored and overwritten (Type 1); here they are derived from the current `mini_sk` reference, which gives the same result.


In [0]:
base_dim_t5 = (
    base_dim_t4_final
    .join(mini_dim_t4, "mini_sk")
    .select(
        "product_id",
        "product_name",
        "mini_sk",
        F.col("category").alias("current_category"),
        F.col("brand").alias("current_brand"),
    )
)
display(base_dim_t5.orderBy("product_id"))
# Note: product 3 (Coffee Mug) and product 7 (Wall Clock) share mini_sk —
# both point at the same (Home, Ikea, White) attribute group.

### Type 6 — hybrid (Type 2 history + Type 1 current-value column)

Full history exactly like Type 2 (reusing `dim_product_t2_final` from the Type 2 cell above), plus a `current_category` column that carries the latest value on every historical row — so old facts can also be grouped by the current attribute value, while the original columns still show the value as it was at each point in time.

Requires the Type 2 cell above to have been executed (uses `dim_product_t2_final`).


In [0]:
# --- Step 1: latest category per product (from the current versions) ---
current_values = (
    dim_product_t2_final
    .filter(F.col("is_current"))
    .select("product_id", F.col("category").alias("current_category"))
)

# --- Step 2: stamp it onto every row, old versions included ---
dim_product_t6_final = dim_product_t2_final.join(current_values, "product_id", "left")

display(dim_product_t6_final.orderBy("product_id", "product_sk"))
# Note: product 3 shows category=Kitchen on the old row and Home on the new,
# but current_category=Home on BOTH rows. Deleted product 4 has no current
# version, so its historical row keeps current_category = null.

### Type 7 — dual Type 1 + Type 2 (two physical tables)

Both tables are maintained in parallel from the same change batch:

* `dim_product_t1`       (Type 1 cell) → current snapshot for dashboards
* `dim_product_t2_final` (Type 2 cell) → full history for point-in-time joins

* inserts → added to both tables
* updates → overwritten in the Type 1 table; new version row in the Type 2 table
* deletes → removed from the Type 1 table; last version closed in the Type 2 table and kept as history

Requires the Type 1 and Type 2 cells above to have been executed (uses `dim_product_t1` and `dim_product_t2_final`).


In [0]:
print("Type 1 table — current snapshot (dashboards / latest value):")
display(dim_product_t1.orderBy("product_id"))
print("Type 2 table — full history (point-in-time fact joins):")
display(dim_product_t2_final.orderBy("product_id", "product_sk"))

## 5. Review of currently used SCD Types

### The questions SCD Types 0–7 answer

Each SCD type is really an answer to a specific business question about **how attribute values change over time**.
Pick the type by asking: *what do I need to know about this attribute's history, and how far back?*

| Type | The question it answers |
|------|------------------------|
| **Type 0 — Retain original** | *"What was the value when the record was first loaded, and should it ever change?"* <br>The attribute is treated as permanent — original value kept forever, no updates applied (e.g. date of birth, original hire date, national ID). |
| **Type 1 — Overwrite** | *"What is the **current** value?"* <br>Only the latest state matters; history is discarded. Wrong-value corrections and reporting that never needs point-in-time accuracy. |
| **Type 2 — Add new row** | *"What was the value **at a specific point in time**, so I can join facts to the attributes that were in effect when the event happened?"* <br>Full history: every change is a new versioned row with `valid_from` / `valid_to` — point-in-time joins, audits, restating history. |
| **Type 3 — Add new column** | *"What is the current value, and what was it **just before** that?"* <br>One step of history only (current + previous). *"What category is this product in now, and what category did we move it from?"* |
| **Type 4 — Mini-dimension** | *"Which combination of rapidly changing attributes was in effect over time — without exploding the base dimension with thousands of rows per entity?"* <br>Groups of attribute values become their own dimension; history lives in the fact table's `mini_sk` references. |
| **Type 5 — Mini-dim + Type 1 outrigger** | *"What is the **current** group of rapid attributes directly from the base dimension, while still being able to reconstruct **past** groupings?"* <br>Type 4 history plus a Type 1 copy of the key current attributes, so dashboards can filter by current value without a join through the facts. |
| **Type 6 — Hybrid (1+2)** | *"What was the value **at the time of the fact**, and what is the **latest** value — on the same row?"* <br>Full Type 2 history plus an overwritten current-value column. *"Sales per current category, even though the fact was recorded under the old one."* |
| **Type 7 — Dual (1+2 tables)** | *"Can I have both a fast, simple current snapshot **and** complete history — without one compromising the other?"* <br>Two physical tables maintained in parallel: Type 1 for dashboards/latest values, Type 2 for point-in-time fact joins. |

**The pattern:** as you go from Type 0 → 7, the question grows from
*"never changes"* → *"what is current?"* → *"what was it then?"* → *"then and now, together"* →
*"many attributes changing fast — group them"* → *"fast current reporting plus full history"*.
Each step trades simplicity for richer time-awareness (or, as in Types 6–7, pays with extra structure/tables to get both at once).

Type 8, described below, answers yet another question: *"Can one single row carry the current values **and** the entire history — natively in the Lakehouse?"*

### What problems can't SCD Types 0–7 solve?

Each type answers a time-awareness question, but every one of them forces a **trade-off** — none gives you *current values* and *complete history* together in a **single row**, natively, in the Lakehouse. The gaps that motivate **SCD Type 8**:

**1. Current snapshot + full history always requires duplication.**
Type 1 (overwrite) loses history; Type 2 keeps it only by multiplying rows. To get both you must either maintain **two physical tables** (Type 7 — double storage, double ETL, risk of drift between them) or bolt an outrigger onto a mini-dimension (Type 5 — still multiple objects and joins).

**2. Row explosion for fast-changing dimensions.**
Type 2 adds a new row per change; for rapidly changing attributes this bloats the base dimension to thousands of rows per entity. Type 4 fixes it only by moving history into the fact table (`mini_sk`), which means point-in-time analysis is only possible through fact joins — never directly from the dimension.

**3. History is only one step deep — or needs an explosion.**
Type 3 stores just *previous*; anything deeper is impossible. Type 2/6 history requires **exploding rows and re-joining** to reconstruct a timeline — there is no way to observe an entity's evolution over time directly on one row.

**4. Every version must fit one rigid schema.**
All Types 0–7 assume a single fixed column set for the whole table. A new attribute forces a full schema change and backfilling nulls for historical rows — there is no *elasticity*, i.e. no way for each version of a record to carry the schema that was actual at that time.

**5. "Current" always costs something.**
Type 2 needs an `is_current` flag + filter (every query touches the whole versioned table); Type 6 adds an overwritten column that must be recomputed on every historical row; Type 5 needs an outrigger join. Dashboards can never read plain, pruned, single-row-per-entity current columns while history sits elsewhere in the same object.

**6. No Lakehouse-native single-object design.**
Types 2–7 model history through *separate rows, separate tables, or separate dimensions* — structures inherited from traditional warehouses. None exploits the Lakehouse's ability to embed history in one row via `ARRAY<STRUCT>`, with column pruning for current-value reporting and higher-order functions / `explode()` for history.

**The gap SCD Type 8 fills:**
one row per entity — current values in regular columns (Type 1-style overwrite) **and** the complete version history embedded in a `_scd` array (version, `valid_from`/`valid_to`, values actual at that time) — current reporting never touches `_scd` (column pruning), history is one `explode()` or higher-order function away, deletes are marked by a closed last version instead of a flag, and the `_scd` schema is a free canvas ("infinity") for whatever a business phenomenon requires.

## 6. Concept of SCD Type 8


### Basic characteristics of the Slowly Changing Dimensions type 8:

- Name follows the next available number of the type — and, when rotated, also means "infinity"
- Designed to take advantage of the "Data Lakehouse" architecture
- One row to store the current record and its history
    - Simple navigation from the current context of a given entity
    - "Current" reporting happens without even touching the `_scd` column — column pruning applied in most cases (getting current values)
    - Previous values reached simply via higher-order functions and/or `explode()` to get the history
    - Present row with current values (no flag `is_current`) — historical entries present in ARRAY `_scd`
    - Deletes -> the row is kept for historical reference: the last `_scd` version is closed (`valid_to` < 9999-12-31), and that closed last version — not a flag column — is what marks the entity as deleted
- Uses a `_scd` ARRAY of STRUCT-type elements that contains the history records (no separate history table object in Lakehouse)
    - Each version of a record is an inner ARRAY element holding that version of the record (starting from version 0) — e.g. an element with the attributes, the change version (0-based), and the VALID_FROM / VALID_TO date fields, all without creating separate rows (unlike, say, SCD Type 2)
    - Elasticity (each entity's version keeps its own schema with the values actual at that time)
- The schema of the `_scd` field is a free canvas:
    - A theoretical model can define recommended columns
    - "Infinity" is achieved here through the flexibility of this schema — you can add whatever you consider needed
- The remaining columns in the table are the attributes that may change
- You can use higher-order functions to observe a given business phenomenon over time, according to the attribute values current at each moment
- Each change is an append of a record to the `_scd` object plus an in-place update of the attributes directly in the table


### Type 8 — embedded history (single row with `_scd` array)

One row per entity. Current attribute values live in the regular columns (overwritten in place — Type 1 style). The full version history is embedded in an `_scd` `ARRAY<STRUCT>` column — each element holds one version with its `valid_from` / `valid_to` and the attribute values that were current at that time.

* inserts → new row with `_scd` = [version 0]
* updates → current columns overwritten (Type 1) and a new version appended to `_scd` (version 1, 2, …)
* deletes → last version in `_scd` closed (`valid_to` set); row kept for historical reference — no flag column, the closed last version (`valid_to` < 9999-12-31) is what marks the delete

"Current" reporting never touches `_scd` (column pruning). History is reached via higher-order functions and/or explode(). No `is_current` / `is_active` flag — the regular columns always hold the latest values.


In [0]:
# --- Step 0: unify original rows and change events into one stream ---
# Original products become "initial" events dated 2024-01-01.
initial = dim_product.select(
    "product_id", "product_name", "category", "brand", "color",
    F.lit("initial").alias("change_type"),
    F.to_date(F.lit("2024-01-01")).alias("event_date"),
)

all_events = initial.unionByName(
    changes.select(
        "product_id", "product_name", "category", "brand", "color",
        "change_type",
        F.col("change_date").alias("event_date"),
    )
)

# --- Step 1: build per-product version rows (non-delete events only) ---
# Each version gets a sequential number (0-based) and a valid_from /
# valid_to window derived from the NEXT event for the same product.
w = Window.partitionBy("product_id").orderBy("event_date")

versions = (
    all_events
    .filter(F.col("change_type") != "delete")
    .withColumn("version", F.row_number().over(w) - 1)
    .withColumn("next_date", F.lead("event_date").over(w))
    .withColumn("valid_from", F.col("event_date"))
    .withColumn(
        "valid_to",
        F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))),
    )
)

# --- Step 2: close the last version for deleted products ---
# If a product was deleted, its last version's valid_to is set to
# delete_date - 1 instead of 9999-12-31.
delete_dates = (
    changes
    .filter(F.col("change_type") == "delete")
    .select("product_id", F.col("change_date").alias("delete_date"))
)

versions = (
    versions
    .join(delete_dates, "product_id", "left")
    .withColumn(
        "valid_to",
        F.when(
            F.col("delete_date").isNotNull() & F.col("next_date").isNull(),
            F.date_sub(F.col("delete_date"), 1),
        ).otherwise(F.col("valid_to")),
    )
    .drop("next_date", "delete_date")
)

# --- Step 3: collect versions into _scd arrays (sorted by version) ---
scd_entries = versions.select(
    "product_id",
    F.struct(
        F.col("version").cast("long").alias("version"),
        F.col("valid_from").alias("valid_from"),
        F.col("valid_to").alias("valid_to"),
        F.col("product_name").alias("product_name"),
        F.col("category").alias("category"),
        F.col("brand").alias("brand"),
        F.col("color").alias("color"),
    ).alias("scd_entry"),
)

scd_arrays = (
    scd_entries
    .groupBy("product_id")
    .agg(F.collect_list("scd_entry").alias("_scd"))
    .withColumn("_scd", F.sort_array("_scd"))
)

# --- Step 4: pick the latest version's values as current columns ---
w_latest = Window.partitionBy("product_id").orderBy(F.col("version").desc())

current = (
    versions
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select("product_id", "product_name", "category", "brand", "color")
)

# --- Step 5: assemble the final Type 8 dimension ---
# No flag column: a deleted entity is recognisable by its last _scd
# version being closed (valid_to < 9999-12-31).
dim_product_t8_final = (
    current
    .join(scd_arrays, "product_id")
    .orderBy("product_id")
)

display(dim_product_t8_final.orderBy("product_id"))

# Current snapshot — the regular columns always hold the latest values.
# Without a flag column, deleted entities are recognised via the LAST
# _scd version's valid_to (9999-12-31 = still active) — a higher-order
# function used only in the filter, so the output columns never show _scd.
last_valid_to = F.expr("element_at(_scd, size(_scd)).valid_to")

display(
    dim_product_t8_final
    .filter(last_valid_to == F.to_date(F.lit("9999-12-31")))
    .drop("_scd")
    .orderBy("product_id")
)

# History via explode — every version on its own row
display(
    dim_product_t8_final
    .select("product_id", F.explode(F.col("_scd")).alias("v"))
    .select(
        "product_id",
        F.col("v.version"), F.col("v.valid_from"), F.col("v.valid_to"),
        F.col("v.product_name").alias("hist_product_name"),
        F.col("v.category").alias("hist_category"),
        F.col("v.brand").alias("hist_brand"),
        F.col("v.color").alias("hist_color"),
    )
    .orderBy("product_id", "version")
)

## 7. Sample with the use of MERGE function in Data Lakehouse

### Setup — MERGE examples (Delta Lake)

`MERGE INTO` works on Delta tables only, so each example below first creates its target table in the schema configured in section 0 (parameters) and then applies the incoming batch with one or more MERGE statements. The DataFrames from the cells above are registered as views so the SQL cells can use them as MERGE sources.


In [0]:
dim_product.createOrReplaceTempView("dim_product_v")
changes.createOrReplaceTempView("changes_v")

print("Views ready: dim_product_v, changes_v")

### SCD Type 1 — overwrite via MERGE

- `delete` → `WHEN MATCHED ... THEN DELETE`
- `update` → `WHEN MATCHED ... THEN UPDATE SET` (overwritten in place)
- `insert` → `WHEN NOT MATCHED ... THEN INSERT`


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t1_dim_product AS
SELECT * FROM dim_product_v;

MERGE INTO scd_t1_dim_product AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.product_name = src.product_name,
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color)
  VALUES
  (src.product_id, src.product_name, src.category, src.brand, src.color);

SELECT * FROM scd_t1_dim_product ORDER BY product_id

### SCD Type 2 — add new row via MERGE

MERGE can close the old version, but a new version is a brand-new row that does not exist in the target yet — so the classic Delta pattern is: one MERGE to expire old rows + one insert for new rows.


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t2_dim_product AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,   -- initial surrogate keys
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE             AS is_current
FROM dim_product_v;

-- MERGE: close the current version of every updated / deleted product
MERGE INTO scd_t2_dim_product AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- INSERT: new versions for updates, first versions for inserts
-- Unique surrogate keys and proper valid_from / valid_to windows
INSERT INTO scd_t2_dim_product
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current
FROM event_stream
WHERE change_type IN ('insert', 'update');

SELECT * FROM scd_t2_dim_product ORDER BY product_id, product_sk

### SCD Type 3 — add new column via MERGE

`previous_category` keeps the old value; category takes the new one. `UPDATE SET` can read the target's pre-update value (tgt.category).


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t3_dim_product AS
SELECT *, CAST(NULL AS STRING) AS previous_category
FROM dim_product_v;

MERGE INTO scd_t3_dim_product AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.previous_category = tgt.category,   -- old value preserved
  tgt.product_name = src.product_name,
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color, previous_category)
  VALUES
  (src.product_id, src.product_name, src.category, src.brand, src.color, NULL);

SELECT * FROM scd_t3_dim_product ORDER BY product_id

### SCD Type 4 — mini-dimension via MERGE

Two tables: a mini-dimension (one row per attribute combination) and a base dimension holding stable attributes + the current `mini_sk` reference. A single MERGE handles all three change kinds on the base table: re-point the reference, delete the base row, insert new rows.


### Mini-dimension

The mini-dimension holds every combination of `category`, `brand` and `color` that exists now or arrives in the batch. Surrogate keys are generated with `DENSE_RANK`, because `MERGE` cannot generate new keys for the rows it inserts.


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t4_mini_dim AS
SELECT
  category, brand, color,
  CAST(1000 + DENSE_RANK() OVER (ORDER BY category, brand, color) AS BIGINT) AS mini_sk
FROM (
  SELECT category, brand, color FROM dim_product_v
  UNION
  SELECT category, brand, color FROM changes_v WHERE change_type IN ('insert', 'update')
) AS combos;
CREATE OR REPLACE TABLE scd_t4_base_dim AS
SELECT d.product_id, d.product_name, m.mini_sk
FROM dim_product_v AS d
JOIN scd_t4_mini_dim AS m
  ON d.category = m.category AND d.brand = m.brand AND d.color = m.color;
MERGE INTO scd_t4_base_dim AS tgt
USING (
  SELECT s.product_id, s.product_name, s.change_type, m.mini_sk
  FROM (
    SELECT product_id, product_name, category, brand, color, change_type
    FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
      FROM changes_v
    )
    WHERE rn = 1
  ) AS s
  LEFT JOIN scd_t4_mini_dim AS m   -- LEFT: delete rows carry no attributes
    ON s.category = m.category AND s.brand = m.brand AND s.color = m.color
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET tgt.mini_sk = src.mini_sk
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, mini_sk)
  VALUES (src.product_id, src.product_name, src.mini_sk);
SELECT * FROM scd_t4_mini_dim ORDER BY mini_sk;
SELECT * FROM scd_t4_base_dim ORDER BY product_id

### SCD Type 5 — mini-dimension + Type 1 outrigger via MERGE

Same two tables as Type 4, but the base dimension also carries a Type 1 copy (outrigger) of the mini-dimension attributes — updated in the same MERGE that re-points the `mini_sk` reference.


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t5_mini_dim AS
SELECT
  category, brand, color,
  CAST(1000 + DENSE_RANK() OVER (ORDER BY category, brand, color) AS BIGINT) AS mini_sk
FROM (
  SELECT category, brand, color FROM dim_product_v
  UNION
  SELECT category, brand, color FROM changes_v WHERE change_type IN ('insert', 'update')
) AS combos;

CREATE OR REPLACE TABLE scd_t5_base_dim AS
SELECT
  d.product_id, d.product_name, m.mini_sk,
  m.category AS current_category,
  m.brand    AS current_brand
FROM dim_product_v AS d
JOIN scd_t5_mini_dim AS m
  ON d.category = m.category AND d.brand = m.brand AND d.color = m.color;

MERGE INTO scd_t5_base_dim AS tgt
USING (
  SELECT s.product_id, s.product_name, s.change_type, m.mini_sk, m.category, m.brand
  FROM (
    SELECT product_id, product_name, category, brand, color, change_type
    FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
      FROM changes_v
    )
    WHERE rn = 1
  ) AS s
  LEFT JOIN scd_t5_mini_dim AS m
    ON s.category = m.category AND s.brand = m.brand AND s.color = m.color
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.mini_sk          = src.mini_sk,
  tgt.current_category = src.category,
  tgt.current_brand    = src.brand
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, mini_sk, current_category, current_brand)
  VALUES
  (src.product_id, src.product_name, src.mini_sk, src.category, src.brand);

SELECT * FROM scd_t5_base_dim ORDER BY product_id

### SCD Type 6 — hybrid (Type 2 + Type 1 current-value column) via MERGE

Three steps: close old versions (MERGE), add new versions (insert), then stamp the latest category onto every historical row (second MERGE). A MERGE source cannot be the target table itself, so the current values are materialized into a small helper table first.


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t6_dim_product AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE AS is_current,
  category AS current_category      -- initially current == original
FROM dim_product_v;

-- MERGE 1: close the current version of every updated / deleted product
MERGE INTO scd_t6_dim_product AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- INSERT: new versions (current_category already equals the new value)
INSERT INTO scd_t6_dim_product
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current, current_category)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current,
  category AS current_category
FROM event_stream
WHERE change_type IN ('insert', 'update');

-- Helper table: latest category per product (MERGE source != target)
CREATE OR REPLACE TABLE scd_t6_current_values AS
SELECT product_id, category
FROM scd_t6_dim_product
WHERE is_current = true;

-- MERGE 2: stamp the latest category onto EVERY historical row
MERGE INTO scd_t6_dim_product AS tgt
USING scd_t6_current_values AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED THEN UPDATE SET
  tgt.current_category = src.category
WHEN NOT MATCHED BY SOURCE THEN UPDATE SET
  tgt.current_category = NULL;

SELECT * FROM scd_t6_dim_product ORDER BY product_id, product_sk

### SCD Type 7 — dual Type 1 + Type 2 via MERGE

Two physical tables, each maintained with its own MERGE from the same incoming batch: a current snapshot table (Type 1) and a full history table (Type 2).


In [0]:
%sql
CREATE OR REPLACE TABLE scd_t7_dim_current AS
SELECT * FROM dim_product_v;

CREATE OR REPLACE TABLE scd_t7_dim_history AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE AS is_current
FROM dim_product_v;

-- Type 1 side: overwrite in place / delete / insert
MERGE INTO scd_t7_dim_current AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.product_name = src.product_name,
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color)
  VALUES (src.product_id, src.product_name, src.category, src.brand, src.color);

-- Type 2 side: close old versions ...
MERGE INTO scd_t7_dim_history AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- ... and add the new versions
INSERT INTO scd_t7_dim_history
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current
FROM event_stream
WHERE change_type IN ('insert', 'update');

SELECT * FROM scd_t7_dim_current ORDER BY product_id;

SELECT * FROM scd_t7_dim_history ORDER BY product_id, product_sk%sql
-- =====================================================================
-- SCD TYPE 7 — DUAL TYPE 1 + TYPE 2 via MERGE
-- =====================================================================
-- Two physical tables, each maintained with its own MERGE from the
-- SAME incoming batch: a current snapshot table (Type 1) and a full
-- history table (Type 2).

CREATE OR REPLACE TABLE scd_t7_dim_current AS
SELECT * FROM dim_product_v;

CREATE OR REPLACE TABLE scd_t7_dim_history AS
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id + 100 AS BIGINT) AS product_sk,
  DATE'2024-01-01' AS valid_from,
  DATE'9999-12-31' AS valid_to,
  TRUE AS is_current
FROM dim_product_v;

-- Type 1 side: overwrite in place / delete / insert
MERGE INTO scd_t7_dim_current AS tgt
USING (
  SELECT product_id, product_name, category, brand, color, change_type, change_date
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM changes_v
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  tgt.product_name = src.product_name,
  tgt.category     = src.category,
  tgt.brand        = src.brand,
  tgt.color        = src.color
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  (product_id, product_name, category, brand, color)
  VALUES (src.product_id, src.product_name, src.category, src.brand, src.color);

-- Type 2 side: close old versions ...
MERGE INTO scd_t7_dim_history AS tgt
USING (
  SELECT product_id, change_date
  FROM (
    SELECT product_id, change_date,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date ASC) AS rn
    FROM changes_v WHERE change_type IN ('update', 'delete')
  )
  WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
 AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1);

-- ... and add the new versions
INSERT INTO scd_t7_dim_history
  (product_id, product_name, category, brand, color, product_sk, valid_from, valid_to, is_current)
WITH event_stream AS (
  SELECT product_id, product_name, category, brand, color, change_type, change_date,
    LEAD(change_date) OVER (PARTITION BY product_id ORDER BY change_date) AS next_date,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn_asc
  FROM changes_v
  WHERE change_type IN ('insert', 'update', 'delete')
)
SELECT
  product_id, product_name, category, brand, color,
  CAST(product_id * 1000 + rn_asc + 99 AS BIGINT) AS product_sk,
  change_date AS valid_from,
  COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31') AS valid_to,
  CASE WHEN next_date IS NULL THEN TRUE ELSE FALSE END AS is_current
FROM event_stream
WHERE change_type IN ('insert', 'update');

SELECT * FROM scd_t7_dim_current ORDER BY product_id;

SELECT * FROM scd_t7_dim_history ORDER BY product_id, product_sk

### SCD Type 8 — embedded history via MERGE

One row per entity. Current values in regular columns (Type 1), full version history embedded in an `_scd` `ARRAY<STRUCT>` column.

With multiple change events per product a single MERGE cannot append multiple versions in one pass (each target row matches at most one source row). Instead we pre-compute the complete `_scd` array per product, load it via `CREATE OR REPLACE TABLE`, then demonstrate the incremental MERGE pattern for a single new change.


In [0]:
%sql
-- --- Build the complete final state from the source data ---
CREATE OR REPLACE TABLE scd_t8_dim_product AS
WITH all_events AS (
  SELECT product_id, product_name, category, brand, color,
         'initial'  AS change_type,
         DATE'2024-01-01' AS event_date
  FROM dim_product_v
  UNION ALL
  SELECT product_id, product_name, category, brand, color,
         change_type, change_date AS event_date
  FROM changes_v
),
versions AS (
  SELECT
    product_id, product_name, category, brand, color,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY event_date) - 1 AS version,
    event_date AS valid_from,
    LEAD(event_date) OVER (PARTITION BY product_id ORDER BY event_date) AS next_date
  FROM all_events
  WHERE change_type != 'delete'
),
closed AS (
  SELECT
    v.product_id, v.product_name, v.category, v.brand, v.color,
    v.version, v.valid_from,
    CASE
      WHEN d.delete_date IS NOT NULL AND v.next_date IS NULL
        THEN DATE_SUB(d.delete_date, 1)
      WHEN v.next_date IS NOT NULL
        THEN DATE_SUB(v.next_date, 1)
      ELSE DATE'9999-12-31'
    END AS valid_to
  FROM versions v
  LEFT JOIN (
    SELECT product_id, MAX(change_date) AS delete_date
    FROM changes_v WHERE change_type = 'delete'
    GROUP BY product_id
  ) d ON v.product_id = d.product_id
),
scd_arrays AS (
  SELECT
    product_id,
    ARRAY_SORT(COLLECT_LIST(
      named_struct(
        'version',      CAST(version AS BIGINT),
        'valid_from',    valid_from,
        'valid_to',      valid_to,
        'product_name',  product_name,
        'category',      category,
        'brand',         brand,
        'color',         color
      )
    )) AS _scd
  FROM closed
  GROUP BY product_id
),
current_vals AS (
  SELECT product_id, product_name, category, brand, color
  FROM (
    SELECT product_id, product_name, category, brand, color,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY version DESC) AS rn
    FROM closed
  ) WHERE rn = 1
)
SELECT
  c.product_id, c.product_name, c.category, c.brand, c.color,
  s._scd
FROM current_vals c
JOIN scd_arrays   s ON c.product_id = s.product_id
ORDER BY c.product_id;

-- --- Incremental MERGE: add a single new product (demonstration) ---
-- For real incremental loads, process one change batch at a time
-- (at most one event per product per batch).
MERGE INTO scd_t8_dim_product AS tgt
USING (
  SELECT 99 AS product_id, 'Stylus Pen' AS product_name,
         'Electronics' AS category, 'Wacom' AS brand,
         'Black' AS color, DATE'2025-01-15' AS change_date
) AS src
  ON tgt.product_id = src.product_id
WHEN NOT MATCHED THEN INSERT
  (product_id, product_name, category, brand, color, _scd)
  VALUES (
    src.product_id, src.product_name, src.category, src.brand, src.color,
    ARRAY(named_struct(
      'version',      CAST(0 AS BIGINT),
      'valid_from',    src.change_date,
      'valid_to',      DATE'9999-12-31',
      'product_name',  src.product_name,
      'category',      src.category,
      'brand',         src.brand,
      'color',         src.color
    ))
  );

SELECT * FROM scd_t8_dim_product ORDER BY product_id;

-- History exploded — every version on its own row
SELECT
  product_id,
  hist.version, hist.valid_from, hist.valid_to,
  hist.product_name AS hist_product_name,
  hist.category     AS hist_category,
  hist.brand        AS hist_brand,
  hist.color        AS hist_color
FROM scd_t8_dim_product
LATERAL VIEW explode(_scd) scd_exploded AS hist
ORDER BY product_id, hist.version

## 8. Point-in-time query — SCD Type 8

Explode the `_scd` array and filter on `valid_from` / `valid_to` to reconstruct the dimension table as it looked at any chosen date. No separate history table, no complex join — just column access into the embedded history.

### Point-in-time query — "what did the dimension look like on date X?"

SCD Type 8's `_scd` array makes point-in-time queries trivial: explode the `_scd` array, filter to the version whose `valid_from` / `valid_to` window contains the target date, and you get the exact attribute values every product had at that moment.

We pick 2024-09-15 — a date that falls between several change waves, so products are at different versions (some at v0, some at v1, v2, …).


In [0]:
target_date = "2024-09-15"

point_in_time = (
    dim_product_t8_final
    .select("product_id", F.explode(F.col("_scd")).alias("v"))
    .filter(
        (F.col("v.valid_from") <= F.lit(target_date).cast("date"))
        & (F.col("v.valid_to") >= F.lit(target_date).cast("date"))
    )
    .select(
        "product_id",
        F.col("v.version"),
        F.col("v.product_name"),
        F.col("v.category"),
        F.col("v.brand"),
        F.col("v.color"),
        F.col("v.valid_from"),
        F.col("v.valid_to"),
    )
    .orderBy("product_id")
)

print(f"Dimension state on {target_date}:")
display(point_in_time)

# 9. Slowly Changing Dimensions (SCD) - comparison of announced Types with the concept one

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice, and an eighth type extends the concept for the Data Lakehouse. Each type answers the question: *when an attribute value changes, what do we do with the old value?*

---

| Type | Name | Strategy | Year announced | Old value preserved? | Full history? | New objects created |
|------|------|---------|---------------|---------------------|---------------|--------------------|
| 0 | Retain / Original | Never change the value | 1996 (2008*) | ✅ (never overwritten) | ❌ | None — no schema or table changes |
| 1 | Overwrite | Replace old with new | 1996 | ❌ | ❌ | None — same table, same columns, in-place updates |
| 2 | Add New Row | Insert a new row, close out the old | 1996 | ✅ | ✅ | **New columns** added to the existing dimension: `valid_from`, `valid_to`, `is_current`, plus a **surrogate key** column (e.g. `product_sk`) |
| 3 | Add New Column | Keep old + new in separate columns | 1996 | ✅ (one prior version) | Partial (current + previous) | **New columns** added to the existing dimension (e.g. `previous_category`) — one per tracked attribute |
| 4 | Mini-Dimension | Rapidly changing attrs split into a separate dimension | 2002 | N/A (separate table) | ✅ (via mini-dim grouping) | **New table** (mini-dimension) for the rapidly-changing attributes, keyed by a surrogate key; base dimension gains a **new FK column** (e.g. `mini_sk`) pointing to it |
| 5 | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | 2002 | Partial | ✅ (via mini-dim) | Everything from Type 4 **plus new outrigger columns** copied into the base dimension (e.g. `current_category`, `current_brand`) carrying the latest mini-dim values |
| 6 | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | 2002 | Partial | ✅ | All Type 2 columns **plus a new current-value column** (e.g. `current_category`) stamped across every history row |
| 7 | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | 2013 | Partial | ✅ | **Second physical table** — a Type 1 snapshot table alongside the Type 2 history table (two tables maintained in parallel) |
| 8 | Embedded History (∞) | One row per entity; full history in `ARRAY<STRUCT>` `_scd` column | 2026 | ✅ | ✅ | **New column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension — no new tables; each array element holds one version with attributes, version number, and `valid_from`/`valid_to` |

---

*announced in 1996 with other concepts, but described in details in 2008.

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

> **New objects:** None. The dimension table and its schema stay exactly as-is; no columns, tables, or lookup structures are added.
>
> **Announced:** Formally described by the Kimball Group in the third edition of *The Data Warehouse Toolkit* (2008), complementing the original three types.
> **Data Warehouse use case:** Preserving immutable attributes such as a customer's original registration date or date of birth that must never be modified.
> **Data Lakehouse use case:** Retaining original source-system landing timestamps in the Bronze layer for full audit traceability across the medallion architecture.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

> **New objects:** None. Same table, same columns — rows are simply overwritten in place.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Correcting a misspelled customer name or keeping the latest shipping address when only the current value matters.
> **Data Lakehouse use case:** Overwriting the latest record state in the Silver/Gold Delta tables via `MERGE` for downstream consumers that need only current values.

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

> **New objects:** The **existing dimension table gains new columns** — at minimum a surrogate key (e.g. `product_sk`), `valid_from`, `valid_to`, and `is_current`. No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Tracking full customer address history so that fact rows join to the dimension version that was active at the time of each transaction.
> **Data Lakehouse use case:** Maintaining full version history in Delta tables using Change Data Feed (CDF) with `MERGE` SCD Type 2 logic for point-in-time analytics and incremental CDC pipelines.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

> **New objects:** **New columns** added to the existing dimension table — one `previous_*` column per tracked attribute (e.g. `previous_category`). No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Comparing current vs. previous sales-territory assignment for an employee without the overhead of full history rows.
> **Data Lakehouse use case:** Lightweight one-level history tracking in a Silver Delta table when full SCD Type 2 is unnecessary but a before/after comparison is still required.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

> **New objects:** A **new lookup table** (the mini-dimension) holding one row per unique attribute combination, keyed by its own surrogate key. The base dimension gains a **new foreign-key column** (e.g. `mini_sk`) referencing the mini-dimension.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) to handle rapidly changing attributes.
> **Data Warehouse use case:** Managing fast-changing customer demographics (income band, credit score, dependents) without exploding the base customer dimension.
> **Data Lakehouse use case:** Splitting volatile attributes into a separate Delta table keyed by surrogate key, referenced from the main dimension to keep the base table compact and queryable.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

> **New objects:** Everything from Type 4 (new mini-dimension table + `mini_sk` FK on the base) **plus new outrigger columns** on the base dimension (e.g. `current_category`, `current_brand`) that carry a Type 1 copy of the latest mini-dimension attribute values.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as an extension of the Type 4 mini-dimension pattern.
> **Data Warehouse use case:** Allowing BI dashboards to filter by the current income band directly from the base dimension while the mini-dimension preserves full historical accuracy.
> **Data Lakehouse use case:** Adding current-value outrigger columns in the Gold layer so BI tools can query volatile attributes without joining to the mini-dimension Delta table.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

> **New objects:** All Type 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`) **plus a new current-value column** (e.g. `current_category`) added to the existing dimension table. No additional tables are created.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as a hybrid combining Types 1 and 2.
> **Data Warehouse use case:** Grouping all historical sales by the *current* product category while still preserving the category value as it was at each point in time.
> **Data Lakehouse use case:** Stamping a `current_*` column across every history row in a Delta table so a single Gold-layer query supports both current and point-in-time analytics.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

> **New objects:** A **second physical table** — a Type 1 snapshot table alongside the Type 2 history table. The Type 2 table carries the SCD 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`); the Type 1 table mirrors the original dimension schema. Two tables must be maintained and kept in sync.
>
> **Announced:** Type 7 was formally described in 2013 by the Kimball Group. It was included in the 3rd edition of The Data Warehouse Toolkit the same year.
> **Data Warehouse use case:** Serving a Type 1 current-state table for fast dashboards and a Type 2 history table for accurate point-in-time fact joins from the same source feed.
> **Data Lakehouse use case:** Maintaining a current-state Delta table (feeding BI/SQL warehouses) alongside a full-history Delta table (feeding incremental CDC and audit pipelines), both refreshed from the same batch or streaming source.

### Type 8 — Embedded History (∞)
One row per entity stores **both current and historical data**. Current attribute values live in scalar columns (no `is_current` flag); the full version history is embedded in an `_scd` `ARRAY<STRUCT>` column where each element holds one version with its attributes, version number, and `valid_from` / `valid_to` timestamps. The name "8" also represents infinity (∞) when rotated, reflecting the unlimited schema flexibility of the embedded history field. Designed for the **Data Lakehouse** architecture: column pruning serves current-value queries without touching `_scd`, while higher-order functions and `explode()` access historical versions for point-in-time analysis. Each change appends a record to the `_scd` array and updates the scalar columns in place — no new rows, no separate history table.

> **New objects:** A **new column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension table. No additional tables are created — the full history lives inside the single row per entity.
>
> **Announced:** Designed for the Data Lakehouse architecture; the name follows the next available SCD type number and, when rotated, represents infinity (∞).
> **Data Warehouse use case:** Not traditionally applicable — this type was designed specifically for Lakehouse-native patterns.
> **Data Lakehouse use case:** Maintaining one row per entity in a Delta table with embedded history, enabling both current-value queries (via column pruning) and point-in-time analytics (via array functions) without a separate history table or multi-row joins.

# 10. SCD Type 8 vs. Data Vault 2.0 — audit history of records

Both SCD Type 8 and Data Vault 2.0 answer the same core question — *"how do we keep the full history of what an attribute looked like, when it changed, and without ever losing the old value?
"* — but they operate at very different levels of the architecture.

### Data Vault 2.0 — where the audit history lives

Data Vault 2.0 splits a model into **Hubs** (business keys), **Links** (relationships), and **Satellites** (all descriptive attributes + the audit trail). The audit history is carried by 
Satellites:

- **`load_date` / `load_end_date`** — each Satellite row is a "version" of the record, opened and closed exactly like SCD Type 2's `valid_from` / `valid_to`, one row per change event.
- **`hash_diff`** — a hash of the attribute payload that is recomputed on each load; a change is detected by comparing hashes, not column-by-column (mirrors how Delta's `MERGE` can match on 
a hash to decide whether anything changed).
- **`record_source`** — provenance of every version (which system, which feed) — something the `_scd` array does *not* carry by default.
- **Point-in-Time (PIT) tables** and **current-state views** over the Raw Vault reconstruct "what the entity looked like on date X" or "the latest values" — the same jobs done by SCD Type 
8's `explode()` on `_scd` and plain column reads respectively.
- The **Business Vault** layer materializes the dimensional/SCD-style outputs (including SCD Type 2 tables) on top of the raw audit history.

### Side-by-side comparison

| Aspect | SCD Type 8 (embedded history) | Data Vault 2.0 (Satellite-based) |
|---|---|---|
| Audit granularity | One row per entity; every change is an element appended to the `ARRAY<STRUCT>` `_scd` column | One row per change event per Satellite; history grows as separate rows (hub–link–satellite structure) |
| Change detection | Per-attribute or hash comparison inside `MERGE` logic | `hash_diff` comparison — built into the standard load pattern |
| Point-in-time query | `explode(_scd)` + filter `valid_from`/`valid_to` in a single table scan of one row per entity | Query across Hub → Satellite joins (or PIT table) — more joins, but the model is pre-joined once in the PIT |
| Current values | Plain scalar columns — column pruning, no history touched; fastest possible read | Current-state views over Satellites (or PIT) — extra view/join hop, but still materialized views |
| Provenance / audit metadata | Schema of `_scd` is a free canvas — version number, `valid_from`/`valid_to` included, but `record_source` and load timestamps are not standardized; you add them if needed | Audit is standardized by design: `load_date`, `hash_diff`, `record_source` on every Satellite row |
| Schema evolution | Elastic at the array level — all elements share ONE struct type, so adding a field widens every element (older versions read it as NULL); a single version cannot have its own set of fields | Satellites evolve by adding columns; old rows keep `NULL` for new attributes |
| Model scope | A single dimension table — a tactical pattern for one table | An entire enterprise modeling methodology — normalized raw layer + agile delivery of marts |
| Storage / update cost | Each change = rewrite of the `_scd` array + in-place attribute update in one row | Each change = simple append of one Satellite row (insert-only Raw Vault loads, parallelizable) |
| Deletes | Must be modeled explicitly (closure of the last `_scd` version's `valid_to`), as done in this notebook | Not modeled in Raw Vault — deletes arrive as new Satellite rows or are resolved in the Business Vault |
| Rebuilding dimensional marts | Direct — the dimension *is* the history | Business Vault / `MERGE` jobs generate SCD 1/2/6/8-style marts from the Vault on demand |

### Takeaways

- **Complementary, not competing.** Data Vault 2.0 is a full modeling methodology whose Satellites are effectively a *de-normalized-at-the-record* audit history; SCD Type 8 is a compact, 
Lakehouse-native way to embed the *resulting* history into one row of the consuming dimension.
- In a Lakehouse, a natural pattern is: **Raw Vault Satellites as the system-of-record audit layer**, then materialize SCD Type 8 dimensions (`_scd` arrays) into the Gold layer from that 
history — you get standardized provenance and audit metadata *and* fast current-value / point-in-time access in the dimensional layer.
- Where SCD Type 8 wins: simplicity (one table, no joins), column pruning for current-value queries, elastic per-version schemas, and trivial point-in-time via `explode()`.
- Where Data Vault 2.0 wins: standardized audit metadata (`record_source`, `load_date`, `hash_diff`), insert-only parallel loads, scaling to enterprise-wide models with many sources, and 
on-demand generation of *any* SCD type (including Type 8) as a Business Vault deliverable.

# 11. Case: schema evolution — new fields added to the dimension

Schemas are never frozen: one day the source system starts sending a **`supplier`**
attribute, later a **`warranty_months`** one — fields that simply did not exist when
the dimension was first built. This section simulates a **second feed** whose
events introduce new attributes mid-stream:

1. **2025-01-10** — the field `supplier` appears in the source for the first time (updates + a brand-new insert that already carries it)
2. **2025-02-15 / 2025-02-20** — a second new field, `warranty_months`, appears

For **SCD Type 8** the case is executed in full: the feed is applied to the
`dim_product_t8_final` table from section 6 and the result shows how the `_scd`
column **behaves and evolves** — the `ARRAY<STRUCT>` schema is widened in place:

* every **existing** version element keeps "the fields that were actual at its
  time" — the new fields stay `NULL` for versions created before the field existed
  (schema **elasticity**),
* every **new** version element carries the fields that exist when it is created,
* the regular (current) columns of the table gain the new attributes too —
  schema evolution of the table itself, with no backfill of history and no second
  physical table.

For **Types 0–7 only a summary** is given (next cell): there the new fields force a
flat, table-wide schema change, because all versions share one column set.

### How Types 0–7 handle this schema evolution (summary only)

All classic types store versions in **one flat schema** — rows or columns of a
single table — so a new field is always a table-wide change; old versions keep
`NULL` and cannot express "this attribute did not exist back then".

| Type | How the new fields (`supplier`, `warranty_months`) would be handled |
|------|---------------------------------------------------------------------|
| **0 — Retain** | `ALTER TABLE … ADD COLUMN` once. Existing rows keep `NULL` forever (Type 0 never rewrites anything); only records inserted after the change can carry a value. |
| **1 — Overwrite** | `ALTER TABLE … ADD COLUMN`; existing rows are `NULL` until the next update/insert overwrites them, or a one-off backfill `UPDATE` fills them. A current-state table — history was never kept anyway. |
| **2 — Add new row** | Add the columns to the versioned table. Old version rows keep `NULL` — which here is actually correct: the field did not exist then. New version rows carry values. The load MERGE runs with schema evolution (`spark.databricks.delta.schema.autoMerge.enabled = true`) or an explicit `ALTER TABLE` first. |
| **3 — Add new column** | The new attribute becomes just another current column (`supplier`); if the *previous* value must also be kept, a `previous_supplier` column is added alongside — one more column per tracked attribute, one level of history only. |
| **4 — Mini-dimension** | First decide *where the field belongs*: rapidly changing → the mini-dimension gains the attribute (every combination is re-keyed, `mini_sk` values change); stable → the base dimension gains a plain column. |
| **5 — Mini-dim + Type 1 outrigger** | The Type 4 decision **plus**, for mini-dimension fields, a new outrigger column in the base dimension (e.g. `current_supplier`) carrying the latest mini-dim value. |
| **6 — Hybrid (1 + 2)** | Type 2 handling for history rows (old versions `NULL`, new versions valued) **plus** the new current-value column (`current_supplier`) stamped across *every* history row on the next load. |
| **7 — Dual (1 + 2)** | Both physical tables must be altered and backfilled **in sync** — double schema change, double backfill, risk of drift between the snapshot and history tables. |

**The contrast with Type 8:** in Types 0–7 the schema is a property of the
*table* — one shape for all versions, so a new field is a table-wide `ALTER`.
In Type 8 the schema is a property of the *version*: each `_scd` element keeps
the fields that existed when that version was created, so history is never
rewritten when a new field arrives.

### Schema evolution feed — new fields appear mid-stream

A second change feed arrives. Unlike the 2024 feed, these events carry new attributes that did not exist when the dimension was built:

* 2025-01-10 → "supplier" appears in the source for the first time
* 2025-02-15+ → "`warranty_months`" appears for the first time

Events created before a field existed simply do not carry it (NULL) — exactly how a real source behaves when new columns are switched on. No delete events here — the focus is purely on schema growth.


In [0]:
schema_evolution_events = spark.createDataFrame([
    # --- wave 1: NEW FIELD supplier (2025-01-10) -----------------------
    Row(product_id=1,  product_name="Laptop Pro",  category="Computers",   brand="Acer",       color="Graphite", supplier="TechSource",   warranty_months=None, change_type="update", 
    change_date="2025-01-10"),
    Row(product_id=3,  product_name="Coffee Mug", category="Home",        brand="Bodum",      color="Ivory",    supplier="HomeGoods Co", warranty_months=None, change_type="update", 
    change_date="2025-01-10"),
    Row(product_id=8,  product_name="Smart Watch", category="Electronics", brand="Apple",      color="Black",    supplier="AppleDist",   warranty_months=None, change_type="update", 
    change_date="2025-01-10"),
    Row(product_id=30, product_name="E-Reader",    category="Electronics", brand="Kobo",       color="Black",    supplier="TechSource",   warranty_months=None, change_type="insert", 
    change_date="2025-01-10"),
    # --- wave 2: NEW FIELD warranty_months (2025-02-15+) ---------------
    Row(product_id=2,  product_name="Headphones", category="Audio",       brand="Sennheiser", color="Navy",     supplier="AudioPlus",    warranty_months=24,    change_type="update", 
    change_date="2025-02-15"),
    Row(product_id=30, product_name="E-Reader",    category="Electronics", brand="Kobo",       color="Sand",     supplier="BookDist",    warranty_months=12,    change_type="update", 
    change_date="2025-02-15"),
    Row(product_id=1,  product_name="Laptop Pro",  category="Computers",   brand="Acer",       color="Graphite", supplier="TechSource",   warranty_months=36,    change_type="update", 
    change_date="2025-02-20"),
])

display(schema_evolution_events.orderBy("change_date", "product_id"))

### Type 8 + schema evolution — applying the new-fields feed

1. The existing `_scd` `ARRAY<STRUCT>` is widened in place: every existing version element gains supplier / `warranty_months` as NULL — those fields did not exist at that version's time (elasticity).
2. The feed creates new version elements that carry the new fields.
3. The previous last version of each touched product is closed (`valid_to` = next `change_date` - 1), versions keep numbering 0,1,2,…
4. The regular (current) columns also gain the new attributes — the table itself evolves schema, no backfill, no flag column.


In [0]:
print("_scd struct BEFORE schema evolution (no supplier / warranty_months):")
dim_product_t8_final.printSchema()

# --- Step 1: widen every existing _scd element with the new fields ----
t8_base = dim_product_t8_final.withColumn(
    "_scd",
    F.transform(
        F.col("_scd"),
        lambda e: F.struct(
            e["version"].cast("long").alias("version"),
            e["valid_from"].alias("valid_from"),
            e["valid_to"].alias("valid_to"),
            e["product_name"].alias("product_name"),
            e["category"].alias("category"),
            e["brand"].alias("brand"),
            e["color"].alias("color"),
            F.lit(None).cast("string").alias("supplier"),
            F.lit(None).cast("long").alias("warranty_months"),
        ),
    ),
)

baseline_versions = (
    t8_base.select("product_id", F.explode("_scd").alias("v"))
    .select("product_id", "v.*")
)

# --- Step 2: new versions from the evolving feed ----------------------
w_new = Window.partitionBy("product_id").orderBy("change_date")

# Guard: an event is applied only if (a) the product is NEW (not in the baseline),
# or (b) its last baseline version is still OPEN and the event is dated after
# that version started. Events for products that were already deleted would
# silently reopen a closed entity (and a replayed feed would duplicate
# versions), so they are skipped and counted instead.
open_base = (
    baseline_versions
    .filter(F.col("valid_to") == F.to_date(F.lit("9999-12-31")))
    .select("product_id", F.col("valid_from").alias("open_from"))
)
known_base = baseline_versions.select("product_id").distinct().withColumn("is_known", F.lit(True))

events_in = (
    schema_evolution_events
    .filter(F.col("change_type") != "delete")
    .withColumn("change_date", F.to_date("change_date"))
)
events_ok = (
    events_in
    .join(known_base, "product_id", "left")
    .join(open_base, "product_id", "left")
    .filter(F.col("is_known").isNull()
            | (F.col("open_from").isNotNull() & (F.col("change_date") > F.col("open_from"))))
    .drop("is_known", "open_from")
)
print(f"Feed events: {events_in.count()}, applied: {events_ok.count()}, "
      f"skipped by the guard (deleted product / replay): {events_in.count() - events_ok.count()}")

new_versions = (
    events_ok
    .withColumn("rn_new", F.row_number().over(w_new))
    .withColumn("next_date", F.lead("change_date").over(w_new))
    .withColumn("valid_from", F.col("change_date"))
    .withColumn("valid_to", F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))))
    .join(t8_base.select("product_id", F.size("_scd").alias("n_base")), "product_id", "left")
    .withColumn("version", F.coalesce(F.col("n_base"), F.lit(0)) + F.col("rn_new") - 1)
)

# --- Step 3: close the previous last version where new events arrive --
first_new_date = (
    new_versions.groupBy("product_id")
    .agg(F.min("valid_from").alias("first_new_date"))
)

baseline_closed = (
    baseline_versions
    .join(first_new_date, "product_id", "left")
    .withColumn(
        "valid_to",
        F.when(
            F.col("first_new_date").isNotNull()
            & (F.col("valid_to") == F.to_date(F.lit("9999-12-31"))),
            F.date_sub(F.col("first_new_date"), 1),
        ).otherwise(F.col("valid_to")),
    )
)

version_cols = ["product_id", "version", "valid_from", "valid_to",
                "product_name", "category", "brand", "color",
                "supplier", "warranty_months"]

all_versions = baseline_closed.select(*version_cols).unionByName(
    new_versions.select(
        F.col("product_id").cast("long").alias("product_id"),
        F.col("version").cast("long").alias("version"),
        F.col("valid_from"), F.col("valid_to"),
        F.col("product_name"), F.col("category"), F.col("brand"), F.col("color"),
        F.col("supplier").cast("string").alias("supplier"),
        F.col("warranty_months").cast("long").alias("warranty_months"),
    )
)

# --- Step 4: re-assemble _scd arrays and the evolved current columns --
scd_arrays = all_versions.groupBy("product_id").agg(
    F.sort_array(
        F.collect_list(
            F.struct(
                F.col("version").cast("long").alias("version"),
                F.col("valid_from").alias("valid_from"),
                F.col("valid_to").alias("valid_to"),
                F.col("product_name").alias("product_name"),
                F.col("category").alias("category"),
                F.col("brand").alias("brand"),
                F.col("color").alias("color"),
                F.col("supplier").alias("supplier"),
                F.col("warranty_months").alias("warranty_months"),
            )
        )
    ).alias("_scd")
)

w_latest = Window.partitionBy("product_id").orderBy(F.col("version").desc())
current = (
    all_versions
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select("product_id", "product_name", "category", "brand", "color", "supplier", "warranty_months")
)

dim_product_t8_evolved = current.join(scd_arrays, "product_id").orderBy("product_id")

print("\n_scd struct AFTER schema evolution — supplier and warranty_months are now part of every version element:")
dim_product_t8_evolved.printSchema()

# Current snapshot — deleted entities (last _scd version closed, e.g.
# product 12) are recognised via the LAST _scd version's valid_to —
# no flag column, consistent with the Type 8 design rule.
last_valid_to = F.expr("element_at(_scd, size(_scd)).valid_to")

print("\nCurrent snapshot (active entities) — the new attributes are plain columns now (the table itself evolved schema):")
display(
    dim_product_t8_evolved
    .filter(last_valid_to == F.to_date(F.lit("9999-12-31")))
    .drop("_scd")
    .orderBy("product_id")
)

### Persist Type 8 + schema evolution to A real Delta table

1. Baseline: the Type 8 table as it was before the new fields existed.
2. `ALTER TABLE` widens the table and the `_scd` element struct in place (metadata-only change: existing data files are not rewritten).
3. MERGE writes only the rows whose `_scd` array grew (plus new entities).

Every step is a Delta commit, so the table history (chapter 12) records the evolution as well. Change Data Feed is enabled so row-level changes can be compared with the `_scd` array later.


In [0]:
TBL = TBL_T8_EVOLUTION   # from the parameters cell (section 0)

(dim_product_t8_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(TBL))

spark.sql(f"ALTER TABLE {TBL} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {TBL} ADD COLUMNS (supplier STRING, warranty_months BIGINT)")
spark.sql(f"ALTER TABLE {TBL} ADD COLUMNS (_scd.element.supplier STRING, _scd.element.warranty_months BIGINT)")

dim_product_t8_evolved.createOrReplaceTempView("t8_evolved_src")
spark.sql(f"""
MERGE INTO {TBL} AS t
USING t8_evolved_src AS s
  ON t.product_id = s.product_id
WHEN MATCHED AND size(s._scd) <> size(t._scd) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.table(TBL).printSchema()
display(spark.sql(f"DESCRIBE HISTORY {TBL}").select("version", "timestamp", "operation", "operationMetrics").orderBy("version"))

### How `_scd` behaves and evolves — version-by-version view (from Delta)

Read the persisted Delta table back and explode the evolved `_scd` for the products touched by the new feed. All elements of the array share one struct type: versions created before the new fields existed read them as NULL (field not known at that time), versions created after carry values.


In [0]:
t8_evo = spark.table(TBL)

print("History of the products touched by the schema-evolution feed:")
display(
    t8_evo
    .filter(F.col("product_id").isin(1, 2, 3, 8, 30))
    .select("product_id", F.explode("_scd").alias("v"))
    .select("product_id", "v.*")
    .orderBy("product_id", "version")
)

print("Product 1 — the raw _scd array: v0–v4 built on the old schema (new fields NULL),")
print("v5 gains supplier (2025-01-10), v6 gains warranty_months (2025-02-20):")
display(t8_evo.filter(F.col("product_id") == 1).select("product_id", "_scd"))

# 12. Delta table history of the SCD tables (types 0–8) vs the `_scd` array

Two different histories exist side by side and are easy to confuse.

| | Delta table history (`DESCRIBE HISTORY`, time travel, Change Data Feed) | `_scd` array (Type 8) |
|---|---|---|
| Time axis | **System time**: when the platform committed the change | **Business (valid) time**: when the value was true in the real world (`valid_from` / `valid_to`) |
| Granularity | One entry per commit (a batch of changes) | One element per entity version |
| Written by | Delta, automatically, for every table of every SCD type | The load logic (MERGE / PySpark), on purpose |
| Retention | Limited by log retention (default 30 days) and by `VACUUM` (default 7 days for data files) | As long as the row exists |
| Late-arriving data | Shows *when* it was loaded | Shows *where* it belongs in business time |
| Typical question | "What did the table look like at 10:00 yesterday, and what changed it?" | "What did the dimension look like on 2024-09-15?" |

Sections: **12.A** commit history of all SCD tables (0–8), **12.B** reconciling Delta versions with `_scd` elements, **12.C** example queries that put both side by side, **12.D** retention.

Note: a table that was re-created and re-loaded many times in this notebook keeps all those commits in its history, so commit counts reflect how often the demo was run, not how the SCD type behaves.

### 12.A Delta commit history of the SCD tables (Types 0-8)

Type 0 had no table in section 7 (no MERGE example), so it is created here: insert-only MERGE, updates and deletes are ignored.


In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {TBL_T0} AS SELECT * FROM dim_product_v")
spark.sql(f"""
MERGE INTO {TBL_T0} AS tgt
USING (
  SELECT product_id, product_name, category, brand, color
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY change_date) AS rn
    FROM changes_v WHERE change_type = 'insert'
  ) WHERE rn = 1
) AS src
  ON tgt.product_id = src.product_id
WHEN NOT MATCHED THEN INSERT (product_id, product_name, category, brand, color)
  VALUES (src.product_id, src.product_name, src.category, src.brand, src.color)
""")

SCD_TABLES = {
    "Type 0": [TBL_T0],
    "Type 1": [TBL_T1],
    "Type 2": [TBL_T2],
    "Type 3": [TBL_T3],
    "Type 4": [TBL_T4_BASE, TBL_T4_MINI],
    "Type 5": [TBL_T5_BASE, TBL_T5_MINI],
    "Type 6": [TBL_T6, TBL_T6_CURRENT],
    "Type 7": [TBL_T7_CURRENT, TBL_T7_HISTORY],
    "Type 8": [TBL_T8, TBL_T8_EVOLUTION],
}

def history_of(scd_type, full_name):
    m = F.col("operationMetrics")
    return spark.sql(f"DESCRIBE HISTORY {full_name}").select(
        F.lit(scd_type).alias("scd_type"), F.lit(full_name.split(".")[-1]).alias("table"),
        "version", "timestamp", "operation",
        m["numTargetRowsInserted"].cast("long").alias("rows_inserted"),
        m["numTargetRowsUpdated"].cast("long").alias("rows_updated"),
        m["numTargetRowsDeleted"].cast("long").alias("rows_deleted"),
        m["numOutputRows"].cast("long").alias("rows_written"),
    )

delta_history = reduce(
    DataFrame.unionByName,
    [history_of(t, n) for t, names in SCD_TABLES.items() for n in names],
)
delta_history.createOrReplaceTempView("scd_delta_history")

print("Every commit of every SCD table - one row per Delta version:")
display(delta_history.orderBy("scd_type", "table", "version"))

print("Per table: number of commits and what they did:")
display(
    delta_history.groupBy("scd_type", "table")
    .agg(F.count("*").alias("commits"),
         F.collect_list("operation").alias("operations"),
         F.sum("rows_inserted").alias("rows_inserted"),
         F.sum("rows_updated").alias("rows_updated"),
         F.sum("rows_deleted").alias("rows_deleted"))
    .orderBy("scd_type", "table")
)

### 12.B reconcile the two histories: Delta versions vs `_scd` elements

Delta time travel shows the table as it was after each commit (system time). The `_scd` array shows every version of every entity (business time). The feed loaded in one MERGE contained 7 events, so the number of `_scd` elements must grow by exactly 7 between the Delta version before the MERGE and the version produced by it (both looked up below, not hard-coded).


In [0]:
# The schema-evolution MERGE is the latest MERGE commit of the table; the
# version just before it is the "before" snapshot. Reused by 12.C1-12.C3.
V_AFTER  = (spark.sql(f"DESCRIBE HISTORY {TBL}").where("operation = 'MERGE'")
            .agg(F.max("version")).first()[0])
V_BEFORE = V_AFTER - 1
print(f"Delta versions used below: before = {V_BEFORE}, after (MERGE) = {V_AFTER}")

schema_evolution_events.createOrReplaceTempView("schema_evolution_events_v")

display(spark.sql(f"""
SELECT 'Delta VERSION AS OF {V_BEFORE} (before the MERGE)' AS snapshot,
       count(*) AS entities, sum(size(_scd)) AS scd_elements
FROM {TBL} VERSION AS OF {V_BEFORE}
UNION ALL
SELECT 'Delta VERSION AS OF {V_AFTER} (after the MERGE)', count(*), sum(size(_scd))
FROM {TBL} VERSION AS OF {V_AFTER}
UNION ALL
SELECT 'Events in the schema-evolution feed', count(DISTINCT product_id), count(*)
FROM schema_evolution_events_v
"""))

### 12.C1 row-level changes from the Delta log (Change Data Feed)

CDF was enabled at Delta version 1, the MERGE is version `V_AFTER` (see 12.B). Every changed row appears as `update_preimage` / `update_postimage` / insert with the commit that wrote it. Compare size(`_scd`) before and after: it grows by the number of events the feed carried for that product.


In [0]:
display(spark.sql(f"""
SELECT product_id, _change_type, _commit_version, _commit_timestamp,
       size(_scd) AS scd_elements, supplier, warranty_months
FROM table_changes('{TBL}', {V_AFTER})
ORDER BY product_id, _commit_version, _change_type
"""))

### 12.C2 two time axes: when the change was true vs when it was loaded

The `_scd` elements added by the MERGE carry business dates (`valid_from`, 2025), the Delta commit that wrote them carries system time (today). The same row change, seen on two different axes.


In [0]:
display(spark.sql(f"""
WITH ch AS (SELECT * FROM table_changes('{TBL}', {V_AFTER})),
post AS (SELECT product_id, _commit_version, _commit_timestamp, _scd
         FROM ch WHERE _change_type = 'update_postimage'),
pre AS (SELECT product_id, size(_scd) AS n_before
        FROM ch WHERE _change_type = 'update_preimage'),
new_elements AS (
  SELECT p.product_id, p._commit_version, p._commit_timestamp,
         explode(filter(p._scd, e -> e.version >= pre.n_before)) AS v
  FROM post p JOIN pre ON p.product_id = pre.product_id
)
SELECT product_id, v.version, v.valid_from AS business_valid_from,
       v.valid_to AS business_valid_to, v.supplier, v.warranty_months,
       _commit_version, _commit_timestamp AS system_commit_time
FROM new_elements
ORDER BY product_id, v.version
"""))

### 12.C3 as-of questions: business time (`_scd`) vs system time (Delta)

(a) What was true for product 1 on 2025-01-15? → `_scd` of the current table.
(b) What did the table say about product 1 before the feed was loaded? → Delta time travel (version `V_BEFORE`), last `_scd` element.
(a) can be answered forever; (b) only while that Delta version is retained.


In [0]:
display(spark.sql(f"""
SELECT 'a) business as-of 2025-01-15 (_scd, current table)' AS question,
       product_id, v.version, v.supplier, v.valid_from, v.valid_to
FROM (SELECT product_id, explode(_scd) AS v FROM {TBL})
WHERE product_id = 1
  AND v.valid_from <= DATE'2025-01-15' AND v.valid_to >= DATE'2025-01-15'
UNION ALL
SELECT 'b) system as-of Delta version {V_BEFORE} (before the load): last _scd element',
       product_id, element_at(_scd, -1).version, element_at(_scd, -1).supplier,
       element_at(_scd, -1).valid_from, element_at(_scd, -1).valid_to
FROM {TBL} VERSION AS OF {V_BEFORE}
WHERE product_id = 1
"""))

### 12.D retention: how far back can each history be read?

Delta history only starts when the table was first written here and is limited by log retention / `VACUUM`. `_scd` reaches back as far as the business data does, independent of when the rows were loaded.


In [0]:
oldest_delta = spark.sql(f"DESCRIBE HISTORY {TBL}").agg(F.min("timestamp").alias("oldest_delta_commit"))
oldest_scd = (spark.table(TBL).select(F.explode("_scd").alias("v"))
              .agg(F.min("v.valid_from").alias("oldest_scd_valid_from")))
display(oldest_delta.crossJoin(oldest_scd))

display(spark.sql(f"SHOW TBLPROPERTIES {TBL}"))

## 12.E What each history can and cannot answer

**Delta history / time travel / Change Data Feed** answers: when was the table changed, by which operation, how many rows were inserted, updated or deleted, and what the table looked like after any retained commit. It works the same for every SCD type, because it records what happened to the *table*. It cannot tell you when a change was true in the real world, and it cannot see anything that happened before the table was first written here.

**The `_scd` array** answers: what was true on any business date, in which order the versions occurred, and what a late-arriving change did to the timeline. It is part of the data, so it survives `VACUUM`, log expiry and table re-creation. It cannot tell you when a version was loaded, by which job, or what the table said before a correction, unless you add such fields to the struct (for example `record_source` or a load timestamp).

**Used together** they give a bi-temporal view: `_scd` for business time, the Delta log for system time. A practical audit pattern is to keep the Change Data Feed for the retention window you need and to join its commit metadata to the `_scd` elements that each commit added (see 12.C2).

Types 0–7 keep their history either nowhere (0, 1), in rows (2, 6, 7), in columns (3) or in a second table (4, 5, 7). In all of them the Delta log is the only record of *when* the rows were written; only Type 8 puts the full business timeline into the same row that the Delta log versions.

# 13. Stress test — bulk load of a randomly-sized change feed

Real feeds are not 7 rows. This chapter loads a **random number of incoming
rows (between 1 and 1.5 million)** into the SCD Type 8 dimension — the
pattern that combines every load concern at once — and measures:

* **how long** each phase takes (feed generation, table seed, MERGE loads,
  schema evolution),
* **what resources** the load consumed (rows inserted/updated, bytes and
  files added/removed, deletion vectors, and the scan / rewrite / source-
  materialisation time split — straight from Delta's own commit metrics),
* and anything remarkable **spotted during the transformation**.

The feed carries **all change kinds at once**, split into three waves
(mimicking three source batches):

1. `2025-03-01` — bulk **inserts** of brand-new products (ids ≥ 10,000,000)
2. `2025-04-01` — **updates** of a sample of wave-1 products plus the original 30 (each appended version grows the `_scd` array)
3. `2025-05-01` — **deletes** of a sample (last `_scd` version closed, no flag column) plus **updates carrying a brand-new column** — `stock_qty` — so **schema evolution** happens mid-load, after two waves were already loaded on the old schema

The target is a fresh copy of the Type 8 evolution table
(`scd_t8_stress`, name from the parameters cell), so the run is repeatable
and never touches the demo tables. Timing is wall-clock
(`time.perf_counter`); resource usage comes from Delta's commit metrics
(`DESCRIBE HISTORY` → `operationMetrics`) and `DESCRIBE DETAIL`.

### 13.A stress feed — random size, all change kinds, new column

`N_ROWS` is drawn pseudo-randomly (fixed seed) between 1 and 1.5 million, so every rerun is reproducible; change seed to draw another size. It is split into three waves that a real source might send as three batches: wave 1 (2025-03-01): bulk inserts of brand-new products wave 2 (2025-04-01): update events (multi-version `_scd` arrays grow) wave 3 (2025-05-01): deletes + updates that carry a brand-new column (`stock_qty`) — schema evolution arrives mid-load The feed is cached + counted so that (a) the random sampling cannot be recomputed differently by the MERGEs, and (b) generation is a measurable phase of its own.


In [0]:
SEED         = 42                                      # fixed seed: reruns draw the same size
random.seed(SEED)
N_ROWS       = random.randint(1_000_000, 1_500_000)   # random load size (reproducible)
N_W1_INSERTS = int(N_ROWS * 0.60)
N_W2_UPDATES = int(N_ROWS * 0.25)
N_W3_DELETES = int(N_ROWS * 0.08)
N_W3_UPDATES = N_ROWS - N_W1_INSERTS - N_W2_UPDATES - N_W3_DELETES

print(f"Randomly drawn stress-feed size: {N_ROWS:,} rows")
print(f"  wave 1 (inserts):                 {N_W1_INSERTS:,}")
print(f"  wave 2 (updates):                  {N_W2_UPDATES:,}")
print(f"  wave 3 (deletes):                  {N_W3_DELETES:,}")
print(f"  wave 3 (updates w/ new column):    {N_W3_UPDATES:,}")

CATEGORIES = ["Electronics", "Computers", "Home", "Office", "Sports",
              "Appliances", "Travel", "Audio", "Decor", "Kitchen"]
BRANDS     = ["Acme", "Bose", "Ikea", "Moleskine", "Dyson", "Razer",
              "Apple", "Kobo", "Sonos", "Varidesk", "Bodum", "Sennheiser",
              "Wacom", "TechSource", "AudioPlus", "BookDist"]
COLORS     = ["Silver", "Black", "White", "Red", "Navy", "Graphite",
              "Ivory", "Oak", "Cream", "Sand", "Beige", "Purple", "Blue"]
SUPPLIERS  = ["TechSource", "HomeGoods Co", "AppleDist", "AudioPlus",
              "BookDist", "MegaSupply", "GlobalDist"]

BASE_ID = 10_000_000   # stress-test products start here, far above the demo ids

def pick(values, *key_cols):
    """Deterministic pseudo-random pick from a literal list (1-based)."""
    return F.element_at(
        F.array(*[F.lit(v) for v in values]),
        F.pmod(F.hash(*key_cols), F.lit(len(values))) + F.lit(1),
    )

def one_event_per_product(df):
    """MERGE allows at most one source row per target row — keep one per product."""
    w = Window.partitionBy("product_id").orderBy("_rid")
    return (df.withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1).drop("rn", "_rid"))

t_gen = time.perf_counter()

# --- wave 1: bulk INSERTS of brand-new products (ids 10_000_000 +) ----
w1 = (
    spark.range(0, N_W1_INSERTS, numPartitions=64)
    .select(
        F.col("id").alias("_rid"),
        (F.lit(BASE_ID) + F.col("id")).cast("long").alias("product_id"),
        F.concat(F.lit("Product "), F.col("id").cast("string")).alias("product_name"),
        pick(CATEGORIES, "id").alias("category"),
        pick(BRANDS, "id").alias("brand"),
        pick(COLORS, "id").alias("color"),
        pick(SUPPLIERS, "id").alias("supplier"),
        F.when(F.pmod(F.hash("id"), F.lit(2)) == 0,
               F.pmod(F.hash("id"), F.lit(60)) + F.lit(1)).cast("long").alias("warranty_months"),
        F.lit(None).cast("int").alias("stock_qty"),   # column does not exist yet -> stays NULL
        F.lit("insert").alias("change_type"),
        F.to_date(F.lit("2025-03-01")).alias("change_date"),
    )
    .drop("_rid")
)

# --- wave 2: UPDATEs — 20% hit the original products (1..22 exist in the
#     baseline table), 80% hit wave-1 products ---------------------------
w2 = one_event_per_product(
    spark.range(0, N_W2_UPDATES, numPartitions=64)
    .select(
        F.col("id").alias("_rid"),
        F.when(F.rand() < 0.2, F.pmod(F.hash("id"), F.lit(22)) + F.lit(1))
         .otherwise(F.lit(BASE_ID) + (F.rand() * F.lit(N_W1_INSERTS)).cast("long"))
         .cast("long").alias("product_id"),
        F.concat(F.lit("Product v2 "), F.col("id").cast("string")).alias("product_name"),
        pick(CATEGORIES, "id").alias("category"),
        pick(BRANDS, "id").alias("brand"),
        pick(COLORS, "id").alias("color"),
        pick(SUPPLIERS, "id").alias("supplier"),
        F.lit(None).cast("long").alias("warranty_months"),
        F.lit(None).cast("int").alias("stock_qty"),
        F.lit("update").alias("change_type"),
        F.to_date(F.lit("2025-04-01")).alias("change_date"),
    )
)

# --- wave 3a: DELETEs (wave-1 products only; attributes are irrelevant)
w3_del = one_event_per_product(
    spark.range(0, N_W3_DELETES, numPartitions=32)
    .select(
        F.col("id").alias("_rid"),
        (F.lit(BASE_ID) + (F.rand() * F.lit(N_W1_INSERTS)).cast("long")).cast("long").alias("product_id"),
        F.lit(None).cast("string").alias("product_name"),
        F.lit(None).cast("string").alias("category"),
        F.lit(None).cast("string").alias("brand"),
        F.lit(None).cast("string").alias("color"),
        F.lit(None).cast("string").alias("supplier"),
        F.lit(None).cast("long").alias("warranty_months"),
        F.lit(None).cast("int").alias("stock_qty"),
        F.lit("delete").alias("change_type"),
        F.to_date(F.lit("2025-05-01")).alias("change_date"),
    )
)

# --- wave 3b: UPDATEs carrying the NEW COLUMN stock_qty ----------------
w3_upd = one_event_per_product(
    spark.range(0, N_W3_UPDATES, numPartitions=64)
    .select(
        F.col("id").alias("_rid"),
        F.when(F.rand() < 0.2, F.pmod(F.hash("id"), F.lit(22)) + F.lit(1))
         .otherwise(F.lit(BASE_ID) + (F.rand() * F.lit(N_W1_INSERTS)).cast("long"))
         .cast("long").alias("product_id"),
        F.concat(F.lit("Product v3 "), F.col("id").cast("string")).alias("product_name"),
        pick(CATEGORIES, "id").alias("category"),
        pick(BRANDS, "id").alias("brand"),
        pick(COLORS, "id").alias("color"),
        pick(SUPPLIERS, "id").alias("supplier"),
        F.lit(None).cast("long").alias("warranty_months"),
        (F.pmod(F.hash("id"), F.lit(60)) + F.lit(1)).cast("int").alias("stock_qty"),  # NEW COLUMN's data
        F.lit("update").alias("change_type"),
        F.to_date(F.lit("2025-05-01")).alias("change_date"),
    )
)
# one event per product per wave: drop update events for deleted products
w3_upd = w3_upd.join(w3_del.select("product_id"), "product_id", "left_anti")

# --- materialise (freeze the sampling) + measure generation ------------
w1.cache();     W1_COUNT     = w1.count()
w2.cache();     W2_COUNT     = w2.count()
w3_del.cache(); W3_DEL_COUNT = w3_del.count()
w3_upd.cache(); W3_UPD_COUNT = w3_upd.count()
W3_COUNT = W3_DEL_COUNT + W3_UPD_COUNT
GEN_TIME = time.perf_counter() - t_gen

w1.createOrReplaceTempView("stress_w1")
w2.createOrReplaceTempView("stress_w2")
w3_del.unionByName(w3_upd).createOrReplaceTempView("stress_w3")

print(f"\nEvents after one-event-per-product dedup:")
print(f"  stress_w1 (inserts): {W1_COUNT:,}")
print(f"  stress_w2 (updates): {W2_COUNT:,}")
print(f"  stress_w3 (deletes {W3_DEL_COUNT:,} + updates {W3_UPD_COUNT:,}): {W3_COUNT:,}")
print(f"Feed generation time (incl. materialisation): {GEN_TIME:.1f} s")

print("\nSample of wave 1 (bulk inserts, stock_qty still NULL):")
display(w1.orderBy("product_id").limit(5))
print("Sample of wave 3 updates (carry the new stock_qty column):")
display(w3_upd.orderBy("product_id").limit(5))

### 13.B load & measure — seed, three MERGE waves, schema evolution

Target: a fresh copy of the Type 8 evolution table (repeatable run). One Type 8 incremental MERGE per wave (at most one event per product, as MERGE requires). Wave 3 arrives with a new column → two metadata-only `ALTER TABLE`s widen the table and the `_scd` struct first.


In [0]:
timings = {}

def timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    timings[label] = time.perf_counter() - t0
    print(f"{label}: {timings[label]:.1f} s")
    return out

# --- phase 0: seed the stress table from the 23-row evolution table ----
def _seed():
    spark.sql(f"CREATE OR REPLACE TABLE {TBL_T8_STRESS} AS SELECT * FROM {TBL_T8_EVOLUTION}")
timed("seed — fresh copy of the Type 8 evolution table", _seed)

baseline = spark.table(TBL_T8_STRESS).agg(
    F.count("*").alias("rows"),
    F.sum(F.size("_scd")).alias("scd_elements"),
).collect()[0]
ROWS_BEFORE, ELEMENTS_BEFORE = baseline["rows"], baseline["scd_elements"]
print(f"baseline: {ROWS_BEFORE} rows / {ELEMENTS_BEFORE} _scd elements")

# --- the Type 8 incremental MERGE, generated for the CURRENT schema ----
ATTRS = ["product_name", "category", "brand", "color", "supplier", "warranty_months"]

def type8_merge_sql(view, attrs):
    """Type 8 incremental MERGE: update = close last version + append a new
    element to _scd + overwrite current columns; delete = close the last
    version only (no flag column); insert = new row with _scd=[version 0]."""
    e_keep = ", ".join(f"'{f}', e.{f}" for f in attrs)
    s_keep = ", ".join(f"'{f}', src.{f}" for f in attrs)
    e_closed = (f"named_struct('version', e.version, 'valid_from', e.valid_from, "
                f"'valid_to', date_sub(src.change_date, 1), {e_keep})")
    e_new  = (f"named_struct('version', CAST(size(tgt._scd) AS BIGINT), "
              f"'valid_from', src.change_date, 'valid_to', DATE'9999-12-31', {s_keep})")
    e_zero = (f"named_struct('version', CAST(0 AS BIGINT), "
              f"'valid_from', src.change_date, 'valid_to', DATE'9999-12-31', {s_keep})")
    set_cols = ",\n      ".join(f"tgt.{f} = src.{f}" for f in attrs)
    ins_cols = ", ".join(["product_id"] + attrs + ["_scd"])
    ins_vals = ", ".join(["src.product_id"] + [f"src.{f}" for f in attrs] + [f"array({e_zero})"])
    return f"""
MERGE INTO {TBL_T8_STRESS} AS tgt
USING {view} AS src
  ON tgt.product_id = src.product_id
WHEN MATCHED AND src.change_type = 'delete' THEN UPDATE SET
  tgt._scd = transform(tgt._scd, e -> IF(e.version = size(tgt._scd) - 1, {e_closed}, e))
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
  {set_cols},
  tgt._scd = concat(
    transform(tgt._scd, e -> IF(e.version = size(tgt._scd) - 1, {e_closed}, e)),
    array({e_new})
  )
WHEN NOT MATCHED AND src.change_type != 'delete' THEN INSERT
  ({ins_cols})
  VALUES ({ins_vals})
"""

# --- phase 1: wave 1 — bulk inserts ------------------------------------
timed("wave 1 MERGE — bulk inserts",
      lambda: spark.sql(type8_merge_sql("stress_w1", ATTRS)))

# --- phase 2: wave 2 — updates (existing rows, _scd arrays grow) ------
timed("wave 2 MERGE — updates",
      lambda: spark.sql(type8_merge_sql("stress_w2", ATTRS)))

# --- phase 3: schema evolution arrives (metadata-only, no data rewrite)
def _evolve():
    spark.sql(f"ALTER TABLE {TBL_T8_STRESS} ADD COLUMNS (stock_qty INT)")
    spark.sql(f"ALTER TABLE {TBL_T8_STRESS} ADD COLUMNS (_scd.element.stock_qty INT)")
timed("schema evolution — 2x ALTER TABLE (metadata only)", _evolve)

# --- phase 4: wave 3 — deletes + updates carrying the new column -------
timed("wave 3 MERGE — deletes + updates w/ new column",
      lambda: spark.sql(type8_merge_sql("stress_w3", ATTRS + ["stock_qty"])))

LOAD_TOTAL = sum(v for k, v in timings.items() if "MERGE" in k)
MERGED_ROWS = W1_COUNT + W2_COUNT + W3_COUNT   # events that actually reached the MERGEs (after dedup)
print(f"\nTotal MERGE load time: {LOAD_TOTAL:.1f} s")
print(f"  {MERGED_ROWS / LOAD_TOTAL:,.0f} merged events/s   ({MERGED_ROWS:,} events after one-event-per-product dedup)")
print(f"  {N_ROWS / LOAD_TOTAL:,.0f} generated rows/s  ({N_ROWS:,} rows drawn before dedup)")

### 13.C resources used — Delta commit metrics + physical layout

Delta records per-commit operationMetrics: the three MERGE commits tell us rows inserted/updated/copied, bytes and files added/removed, deletion vectors, and the time split (scan / rewrite / source materialisation). describe detail adds the resulting physical layout. The stress table is re-created by every run of 13.B, so its history keeps the commits of earlier runs; only the latest three MERGE commits are shown.


In [0]:
m = F.col("operationMetrics")

print("Per-MERGE commit metrics (from DESCRIBE HISTORY -> operationMetrics):")
display(
    spark.sql(f"DESCRIBE HISTORY {TBL_T8_STRESS}")
    .where(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc()).limit(3)   # the three MERGE waves of the latest run only
    .select(
        "version",
        F.coalesce(m["numTargetRowsMatchedUpdated"], m["numTargetRowsUpdated"]).cast("long").alias("rows_updated"),
        m["numTargetRowsInserted"].cast("long").alias("rows_inserted"),
        m["numTargetRowsCopied"].cast("long").alias("rows_copied"),
        m["numTargetDeletionVectorsAdded"].cast("long").alias("dvs_added"),
        m["numTargetFilesAdded"].cast("long").alias("files_added"),
        m["numTargetBytesAdded"].cast("long").alias("bytes_added"),
        m["numTargetBytesRemoved"].cast("long").alias("bytes_removed"),
        m["executionTimeMs"].cast("long").alias("exec_ms"),
        m["scanTimeMs"].cast("long").alias("scan_ms"),
        m["rewriteTimeMs"].cast("long").alias("rewrite_ms"),
        m["materializeSourceTimeMs"].cast("long").alias("source_ms"),
    )
    .orderBy("version")
)

detail = spark.sql(f"DESCRIBE DETAIL {TBL_T8_STRESS}").select("numFiles", "sizeInBytes").collect()[0]
print(f"Physical layout after the stress load: {detail['numFiles']} files, "
      f"{detail['sizeInBytes'] / 1e6:,.1f} MB total")

print("\nPopulation shape after the load:")
display(
    spark.table(TBL_T8_STRESS).agg(
        F.count("*").alias("rows_total"),
        F.sum(F.size("_scd")).alias("scd_elements_total"),
        F.max(F.size("_scd")).alias("max_versions_per_product"),
        F.round(F.avg(F.size("_scd")), 3).alias("avg_versions_per_product"),
        F.sum(F.when(F.expr("element_at(_scd, size(_scd)).valid_to") < F.to_date(F.lit("9999-12-31")), F.lit(1)).otherwise(F.lit(0))).alias("rows_deleted (closed last version)"),
    )
)

### 13.D verify — expected vs actual + spot check of `_scd` evolution

Expected row growth: baseline + wave-1 inserts (waves 2-3 only target existing products). Expected `_scd` elements: baseline + inserts + updates (deletes close the last version but add no element).


In [0]:
actual = spark.table(TBL_T8_STRESS).agg(
    F.count("*").alias("rows"),
    F.sum(F.size("_scd")).alias("elements"),
).collect()[0]

expected_rows     = ROWS_BEFORE + W1_COUNT
expected_elements = ELEMENTS_BEFORE + W1_COUNT + W2_COUNT + W3_UPD_COUNT

ok_rows     = actual["rows"] == expected_rows
ok_elements = actual["elements"] == expected_elements
print(f"rows:       expected {expected_rows:,} | actual {actual['rows']:,} | {'OK' if ok_rows else 'MISMATCH'}")
print(f"_scd elems: expected {expected_elements:,} | actual {actual['elements']:,} | {'OK' if ok_elements else 'MISMATCH'}")
assert ok_rows and ok_elements, "stress load did not produce the expected table shape"

print("\nSpot check — products whose LAST version carries the new stock_qty column")
print("(older versions in the same array legitimately read stock_qty as NULL):")
display(
    spark.table(TBL_T8_STRESS)
    .where((F.col("stock_qty").isNotNull()) & (F.size("_scd") >= 3))
    .select("product_id", "product_name", "stock_qty", "_scd")
    .limit(3)
)

print("Schema after the stress load (stock_qty added top-level AND inside _scd):")
spark.table(TBL_T8_STRESS).printSchema()

### 13.E Findings — what the stress test observed

**Reference run** (2026-11-04, serverless compute, SEED = 42, so the load size is reproducible): 1,335,243 generated rows. After the one-event-per-product dedup that MERGE requires, 1,190,421 events reached the MERGEs — wave 1: 801,145 inserts · wave 2: 226,841 updates (deduped from 333,810) · wave 3: 162,435 events (100,097 deletes + 62,338 updates carrying the new `stock_qty` column). Wall-clock times vary a little from run to run; cells 13.A–13.D print the figures of the latest run.

**Timings (wall-clock, reference run)**

| Phase | Time | Notes |
|---|---|---|
| Feed generation (incl. materialisation) | ≈ 13 s | `spark.range` + deterministic hash picks is not the bottleneck |
| Seed (copy of the 23-row evolution table) | ≈ 2 s | |
| Wave 1 MERGE — 801,145 inserts | ≈ 5 s | ≈ 160K inserts/s |
| Wave 2 MERGE — 226,841 updates | ≈ 7 s | |
| Schema evolution (2× ALTER TABLE) | ≈ 2 s | metadata-only, no data rewrite |
| Wave 3 MERGE — 162,435 deletes + updates | ≈ 7 s | |
| Total MERGE load | ≈ 19 s | ≈ 63K merged events/s (≈ 71K generated rows/s before dedup) |

**Resources (Delta commit metrics, DESCRIBE HISTORY, latest three MERGE commits)**

| Commit | Rows | Files added | Bytes added | DVs | MERGE execution time |
|---|---|---|---|---|---|
| wave 1 (inserts) | 801,145 inserted | 8 | 4.3 MB | 0 | ≈ 4.3 s |
| wave 2 (updates) | 226,841 updated | 1 | 4.3 MB | 9 | ≈ 5.5 s |
| wave 3 (deletes + updates) | 162,435 updated | 2 | 3.3 MB | 9 | ≈ 5.6 s |

Final table: 801,168 rows, 1,090,392 `_scd` versions (max 9 versions on a single product), in 1 file of 11.0 MB. Expected row and element counts matched exactly (verified in 13.D). What the notebook does **not** measure: CPU, memory or DBUs — only wall-clock time and Delta commit metrics — so nothing below is a claim about how serverless allocated its resources.

**What was spotted during the transformation**

1. **An update costs several times more than an insert, per row.** Wave 2 spent ≈ 5.5 s of MERGE execution on 226,841 updates (≈ 24 µs per update); wave 1 spent ≈ 4.3 s on 801,145 inserts (≈ 5 µs per insert), so roughly 4–5× per row. Every update closes the previous version and rewrites the row's entire `_scd` array.
2. **Deletes are just cheap updates in Type 8** — a delete only closes the last version's `valid_to` (same UPDATE path, no flag column). History is preserved, never destroyed.
3. **Deletion vectors absorbed the updates** — each update wave added only 1–2 data files and 9 DVs, with bytes_removed = 0: changed rows are logically removed via DVs instead of rewriting whole files. No OPTIMIZE was run in this notebook; the table ended as one 11.0 MB file.
4. **Schema evolution mid-load was a non-event** — about 2 s for two metadata-only ALTERs (top-level `stock_qty` and `_scd.element.stock_qty`) between waves 2 and 3; the wave-3 MERGE then handled the new column inside every `named_struct` it writes. Older versions legitimately read `stock_qty` as NULL (see the 13.D spot check).
5. **History is permanent and verifiable** — every update event became exactly one new `_scd` element: 1,090,392 = 68 (baseline) + 801,145 + 226,841 + 62,338. At ≈ 14 bytes per row (compressed) storage size is not the concern; row width growth as arrays grow for high-churn products is.
6. **No compute tuning was needed** for these volumes on serverless; the feed was cached, so the sampling stayed stable across the MERGEs. This notebook does not measure resource usage, so it makes no claim about autoscaling.

**Takeaway:** at this scale (≈ 1.3M events) the Type 8 pattern ran end to end in well under a minute on serverless in this run: generation and load were each under ≈ 20 s. The incremental MERGE + DV + metadata-only-ALTER combination keeps cost proportional to the changed rows, not to the table size. The pattern's write amplification (whole-array rewrite per update) is visible in the per-row cost and is the first thing that will hurt if change frequency per entity grows.
> 


# 14. Summary — the SCD types 0–7 revisited, and Type 8 (the `_scd` array) evaluated

## 14.A What the notebook demonstrated about the classic types (0–7)

| Type | Where history lives | Key finding in this notebook | Main cost |
|---|---|---|---|
| **0 – fixed** | Nowhere (by design) | Insert-only MERGE: updates and deletes in the feed are simply ignored; the first value survives forever | Cannot answer *any* as-of question |
| **1 – overwrite** | Nowhere; the row is the entity | Every MERGE collapses to the latest values; the Delta log is the only record that anything changed | Complete amnesia once retention/VACUUM passes |
| **2 – row versioning** | Extra rows (one per change) | `valid_from` / `valid_to` / `is_current` give correct as-of joins; the demo showed current + history views split naturally | Table grows with every attribute change; all queries must filter one row per key |
| **3 – in-place history columns** | Columns on the same row (`previous_*`) | Keeps one row per entity (no fan-out) but stores only ONE step back — a second change overwrites the earlier "previous" value | Truncated history; schema must be pre-designed |
| **4 – hybrid (SCD2 + mini-dim)** | History in the base dim, volatile attributes in a mini-dim | Slowly vs. rapidly changing attributes are separated; the mini-dim absorbs churn the base table never sees | Two tables + surrogate-key joins on every fact |
| **5 – fast-expanding (mini-dim only)** | Mini-dim attribute groups with fact-group keys | Facts never join the dimension directly; attribute-group versions live in a separate key space | No single-row view of an entity; complex modeling |
| **6 – current + history table** | Regular snapshot table + full history table | Simple "current" queries stay fast, history is complete in the second table | History is a separate object to build, sync and govern |
| **7 – mixed (Type 1 + Type 6 columns)** | Flagged columns / current + history tables | `is_current` plus the classic Type 6 pattern: deterministic which columns track history and which don't | The `is_current` flag forces the two-step delete-then-reinsert pattern and is easy to forget in BI queries |

Cross-cutting findings:

* **None of types 0–7 stores history inside the row itself** — it lives in extra rows, extra columns, or extra tables. Type 8 is the only type where one row carries the entity's complete 
business timeline as an `_scd` ARRAY<STRUCT>.
* **All types depend on the Delta log** for *system-time* questions (when a row was written). Chapter 12 showed the two time axes (business `valid_from`/`valid_to` vs. commit timestamps / 
Change Data Feed) and how they reconcile: the schema-evolution feed grew `sum(size(_scd))` by exactly the number of events between the Delta version before and after the schema-evolution MERGE.
* **Schema evolution** (chapter 11): the Type 8 feed introduced `supplier` and `warranty_months` mid-stream. A metadata-only `ALTER TABLE ... ADD COLUMNS` (including `_scd.
element.*`) widened the struct in place — older versions read the new fields as NULL ("not known at that time"), newer versions carry values. No backfill, no flag columns, no table rewrite.
* **Deletes without a flag column**: a deleted entity is recognized because the LAST `_scd` version's `valid_to` is no longer `9999-12-31` — history is preserved instead of destroyed.

## 14.B Type 8 (`_scd` array) — pros and cons

**Pros**

1. **Complete history in one row** — the full version timeline (values, `valid_from`, `valid_to`, `version`) travels with the entity; no fan-out, no second table, no duplicated keys in the 
current snapshot.
2. **Native as-of queries** — `filter(_scd, e -> e.valid_from <= d AND e.valid_to >= d)` answers "what was true on date d" with a single array predicate.
3. **Elastic schema** — the struct grows by adding fields (top-level columns *and* `_scd.element.*`); old versions legitimately hold NULLs for fields that did not exist yet, which is the 
honest representation.
4. **Metadata-only evolution** — Delta's `ALTER TABLE ADD COLUMNS` rewrites no data files; the chapter-12 history records the evolution as ordinary commits.
5. **Compact and Delta-friendly incremental loads** — the MERGE touches only rows whose `_scd` grew (`size(s._scd) <> size(t._scd)`), and Change Data Feed exposes exactly which entities 
evolved.
6. **Survives VACUUM and log retention** — business history is data, not log; it outlives the log-retention and VACUUM windows that bound time travel (Delta defaults: delta.logRetentionDuration = 30 days, delta.deletedFileRetentionDuration = 7 days; 12.D shows the values set on this table).
7. **No `is_current` flag to forget** — currency is derived (`element_at(_scd, size(_scd)).valid_to = DATE'9999-12-31'`), so there is no is_current flag that can drift out of sync with the dates (closed/deleted entities still sit in the table and are recognised by the last valid_to, so a plain read of the current columns also returns them unless you filter).

**Cons**

1. **Array-of-struct cost** — every row carries its whole history; entities with hundreds of versions bloat rows and degrade column pruning on `_scd`.
2. **No engine-level SCD helper** — the merge logic (close previous version, number versions, rebuild the array) is hand-written PySpark/SQL; there is no `apply_changes`-style automation for 
Type 8.
3. **Rewrites the array on every change** — a one-field update rewrites the entire `_scd` array of that row (arrays grow by whole new elements); write amplification is worse than Type 1's 
in-place update.
4. **Struct-wide schema rigidity** — all array elements share ONE struct type; you cannot drop or retype a field without rewriting, and every version element carries every field.
5. **Awkward analytics** — tools that cannot `explode` arrays need a view layer; fact joins against a specific version always require the explode/filter pattern, which is more work than a 
plain key join.
6. **No per-attribute granularity** — elements store full row images per version, not per-attribute changes; a Type 4-style split of volatile vs. stable attributes is not possible inside one 
`_scd` element.
7. **System-time still external** — Type 8 records *business* time only; "when was this version loaded" still requires the Delta log / CDF (chapter 12), so the bi-temporal picture always 
spans two systems.

## 14.C When to use Type 8 — use cases

**Strong fits**

* **Reference/master data with moderate change frequency** (products, customers, accounts) where the full timeline must be queryable from the current table — a few dozen versions per entity, 
not thousands.
* **As-of reporting in place** — "what was this entity's state on 2024-09-15?" answered without a second table or time travel.
* **Frequently evolving source schemas** — feeds that switch on new attributes mid-stream; the struct widens with NULL-padded history instead of triggering backfills or table rewrites.
* **Retention beyond the Delta log** — compliance or audit needs where business history must survive `VACUUM` and log expiry.
* **Portable history** — one table export carries the whole timeline to another environment; no dependency on a source system's history table.
* **Teaching and small dimensions** — where single-object simplicity outweighs write amplification.

**Weak fits (prefer another type)**

* **High-churn, volatile attributes** (prices, balances, click-stream state) — Type 5's mini-dim or a fact-with-versions model keeps the dimension row stable.
* **Very large dimensions with heavy version counts** — Type 2 rows (or Type 6 + history table) prune and compress better than giant arrays.
* **Workloads where BI users join facts by plain surrogate key** — classic Type 2 with `is_current` filtering is simpler for them than explode-and-filter on every query.
* **Attributes that must evolve independently per field** — Type 3-style columns or the Type 4 split offer field-level history the single struct cannot.

**Bottom line**: Type 8 trades write amplification and hand-rolled merge logic for *self-contained, schema-elastic, retention-proof history inside the row*. Paired with the Delta log (system 
time) it gives a bi-temporal dimension in a single table — the pattern of choice when the timeline must outlive the log and travel with the data, and the wrong one when version volume or BI 
join simplicity dominates.

### 15. Clean-up — drop every table created by this notebook (commented out)

Uncomment the block below to drop all tables this notebook created during its run (MERGE examples for types 0–8, the schema-evolution table, and the stress-test table). It iterates `ALL_TABLES` from the parameters cell (section 0), so it always cleans exactly the catalog/schema configured there.


In [0]:
# for t in ALL_TABLES:
#     print(f"Dropping {t}")
#     spark.sql(f"DROP TABLE IF EXISTS {t}")
# print(f"Done — {len(ALL_TABLES)} tables dropped from {catalog_name}.{schema_name}.")